# Standalone RTDP & OD-RTDP
This notebook contains all the core classes from the framework merged into a single file so it can be run completely standalone (e.g., on Google Colab) without needing the `src/` directory.

In [ ]:
from __future__ import annotations


"""Scale-aware numerical comparisons shared by both planners."""

import math


def tied_by_ulp(a: float, b: float, *, ulps: int = 8) -> bool:
    """Return True when two finite values differ only by floating-point noise."""
    if a == b:
        return True
    if not math.isfinite(a) or not math.isfinite(b):
        return False
    tolerance = ulps * max(math.ulp(a), math.ulp(b), math.ulp(1.0))
    return abs(a - b) <= tolerance


def scaled_residual_ratio(
    old_value: float,
    new_value: float,
    *,
    absolute_tolerance: float,
    relative_tolerance: float,
) -> float:
    """Return residual divided by a scale-aware admissible tolerance."""
    residual = abs(new_value - old_value)
    scale = max(1.0, abs(old_value), abs(new_value))
    allowed = absolute_tolerance + relative_tolerance * scale
    if allowed == 0.0:
        return 0.0 if residual == 0.0 else math.inf
    return residual / allowed



"""Small statistical helpers for experiment-size and stability defaults."""

import math
from statistics import NormalDist


def binomial_worst_case_sample_size(
    *, confidence: float, half_width: float
) -> int:
    """Normal-approximation sample size for a worst-case proportion p=0.5.

    This makes the default number of evaluation episodes interpretable: the
    requested confidence interval has approximately ``half_width`` precision
    in the worst case.  Exact intervals are still recommended in analysis.
    """
    if not 0.0 < confidence < 1.0:
        raise ValueError("confidence must be in (0, 1)")
    if not 0.0 < half_width < 1.0:
        raise ValueError("half_width must be in (0, 1)")
    z = NormalDist().inv_cdf(0.5 + confidence / 2.0)
    return max(1, math.ceil((z * z * 0.25) / (half_width * half_width)))


def consecutive_trials_for_detection(
    *, confidence: float, minimum_event_probability: float
) -> int:
    """Trials needed to see at least one event with the given confidence.

    If an unstable trial would occur independently with probability at least
    ``minimum_event_probability``, observing this many consecutive stable
    trials misses it with probability at most ``1-confidence``.  RTDP trials
    are not truly independent, so this remains an empirical stopping rule, but
    its parameters are explicit rather than a bare fixed streak length.
    """
    if not 0.0 < confidence < 1.0:
        raise ValueError("confidence must be in (0, 1)")
    if not 0.0 < minimum_event_probability < 1.0:
        raise ValueError("minimum_event_probability must be in (0, 1)")
    return max(
        1,
        math.ceil(
            math.log(1.0 - confidence)
            / math.log(1.0 - minimum_event_probability)
        ),
    )



"""Derived limits used by planning and policy evaluation.

The old implementation used a fixed ``5 * expected_distance`` multiplier.
This module replaces that hidden constant with an interpretable tail-probability
bound.  For an isolated agent that needs ``d`` successful moves and succeeds
with probability ``q`` on each attempt, we choose the smallest number of
attempts whose Chernoff upper bound on failing to obtain ``d`` successes is at
most ``tail_probability``.

For several agents, the default sequential bound sums the individual bounds.
This is intentionally conservative: it allows enough steps for the agents to
complete one after another, and cycle diagnostics distinguish an improper
policy from a merely slow stochastic execution.
"""

import math
from collections.abc import Iterable


def isolated_success_attempt_bound(
    required_successes: int,
    success_probability: float,
    tail_probability: float,
) -> int:
    """Return a dependency-free conservative attempt bound.

    The Chernoff lower-tail inequality for ``X ~ Binomial(t, q)`` is used:

        P[X <= k] <= exp(-(t*q-k)^2 / (2*t*q))

    with ``k = required_successes - 1``.  The returned integer is the smallest
    ``t`` for which this bound is no larger than ``tail_probability``.
    """
    if required_successes < 0:
        raise ValueError("required_successes cannot be negative")
    if required_successes == 0:
        return 0
    if not 0.0 < success_probability <= 1.0:
        raise ValueError("success_probability must be in (0, 1]")
    if not 0.0 < tail_probability < 1.0:
        raise ValueError("tail_probability must be in (0, 1)")
    if success_probability == 1.0:
        return required_successes

    failure_threshold = required_successes - 1

    def failure_bound(attempts: int) -> float:
        mean = attempts * success_probability
        if failure_threshold >= mean:
            return 1.0
        return math.exp(
            -((mean - failure_threshold) ** 2) / (2.0 * mean)
        )

    lower = max(
        required_successes,
        math.ceil(required_successes / success_probability),
    )
    upper = lower
    while failure_bound(upper) > tail_probability:
        upper *= 2

    while lower < upper:
        middle = (lower + upper) // 2
        if failure_bound(middle) <= tail_probability:
            upper = middle
        else:
            lower = middle + 1

    return lower


def sequential_multi_agent_step_bound(
    distances: Iterable[float],
    success_probability: float,
    tail_probability: float,
) -> int:
    """Return the sum of isolated per-agent stochastic attempt bounds."""
    integer_distances: list[int] = []
    for distance in distances:
        if math.isinf(distance):
            raise ValueError("All distances must be finite")
        if distance < 0:
            raise ValueError("Distances cannot be negative")
        integer_distances.append(math.ceil(distance))

    total = sum(
        isolated_success_attempt_bound(
            distance,
            success_probability,
            tail_probability,
        )
        for distance in integer_distances
    )
    return max(1, total)



"""Low-overhead process-memory measurement for isolated experiment runs."""

from dataclasses import dataclass
import os
import threading
import time

try:
    import psutil
except ImportError:  # pragma: no cover - clear runtime message on user machines
    psutil = None  # type: ignore[assignment]


MIB = 1024 * 1024


@dataclass(frozen=True)
class ResourceSnapshot:
    baseline_rss_mb: float
    peak_rss_mb: float
    peak_rss_delta_mb: float
    memory_limit_mb: float | None
    memory_limit_reached: bool


class ResourceMonitor:
    """Sample process RSS in a daemon thread.

    ``memory_limit_mb`` is interpreted as additional RSS above the value at the
    start of the measured phase.  This makes sequential interactive runs less
    sensitive to the Python interpreter's already-loaded modules.  For final
    benchmarks, each run should still be isolated in a fresh subprocess.
    """

    def __init__(
        self,
        *,
        memory_limit_mb: float | None = None,
        sample_interval_seconds: float = 0.05,
    ) -> None:
        if memory_limit_mb is not None and memory_limit_mb <= 0.0:
            raise ValueError("memory_limit_mb must be positive or None")
        if sample_interval_seconds <= 0.0:
            raise ValueError("sample_interval_seconds must be positive")
        if psutil is None:
            raise RuntimeError(
                "Memory measurement requires psutil. Install it with: "
                "python -m pip install psutil"
            )

        self.memory_limit_mb = memory_limit_mb
        self.sample_interval_seconds = sample_interval_seconds
        self._process = psutil.Process(os.getpid())
        self._baseline_bytes = int(self._process.memory_info().rss)
        self._peak_bytes = self._baseline_bytes
        self._limit_reached = threading.Event()
        self._stop_event = threading.Event()
        self._thread: threading.Thread | None = None

    def _sample_once(self) -> None:
        try:
            rss = int(self._process.memory_info().rss)
        except Exception:
            return
        if rss > self._peak_bytes:
            self._peak_bytes = rss
        if self.memory_limit_mb is not None:
            delta_mb = (rss - self._baseline_bytes) / MIB
            if delta_mb >= self.memory_limit_mb:
                self._limit_reached.set()

    def _run(self) -> None:
        while not self._stop_event.wait(self.sample_interval_seconds):
            self._sample_once()

    def start(self) -> "ResourceMonitor":
        if self._thread is not None:
            return self
        self._sample_once()
        self._thread = threading.Thread(
            target=self._run,
            name="resource-monitor",
            daemon=True,
        )
        self._thread.start()
        return self

    def stop(self) -> ResourceSnapshot:
        self._stop_event.set()
        if self._thread is not None:
            self._thread.join(timeout=max(1.0, 4 * self.sample_interval_seconds))
        self._sample_once()
        return self.snapshot()

    def limit_reached(self) -> bool:
        return self._limit_reached.is_set()

    def snapshot(self) -> ResourceSnapshot:
        baseline_mb = self._baseline_bytes / MIB
        peak_mb = self._peak_bytes / MIB
        return ResourceSnapshot(
            baseline_rss_mb=baseline_mb,
            peak_rss_mb=peak_mb,
            peak_rss_delta_mb=max(0.0, peak_mb - baseline_mb),
            memory_limit_mb=self.memory_limit_mb,
            memory_limit_reached=self.limit_reached(),
        )

    def __enter__(self) -> "ResourceMonitor":
        return self.start()

    def __exit__(self, exc_type, exc, tb) -> None:
        self.stop()



"""
Grid-map and scenario loader for the multi-agent planning project.

Expected project structure:

final_project/
└── maps/
    └── room-64-64-16/
        ├── room-64-64-16.map
        ├── scen/
        │   ├── room-64-64-16-even-1.scen
        │   └── ...
        └── room-64-64-16.pdf   # ignored by this module

The scenario directory can have any name. Scenario files are discovered
recursively under the selected map folder.
"""

from collections import deque
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator, Sequence
import argparse
import re


Position = tuple[int, int]

# Symbols used by the benchmark .map format.
TRAVERSABLE_SYMBOLS = frozenset({".", "G", "S"})
BLOCKED_SYMBOLS = frozenset({"@", "O", "T", "W"})


class MapCreatorError(ValueError):
    """Raised when a map folder, map file, or scenario file is invalid."""


@dataclass(frozen=True)
class GridMap:
    """Static grid geometry loaded from a .map file."""

    name: str
    path: Path
    width: int
    height: int
    grid: tuple[str, ...]
    obstacles: frozenset[Position]
    free_cells: frozenset[Position]

    def is_free(self, position: Position) -> bool:
        return position in self.free_cells

    def in_bounds(self, position: Position) -> bool:
        x, y = position
        return 0 <= x < self.width and 0 <= y < self.height

    def neighbors4(self, position: Position) -> Iterator[Position]:
        """Yield legal four-directional neighbors."""
        x, y = position

        for next_position in (
            (x + 1, y),
            (x - 1, y),
            (x, y + 1),
            (x, y - 1),
        ):
            if next_position in self.free_cells:
                yield next_position


@dataclass(frozen=True)
class ScenarioEntry:
    """One start-goal task read from a .scen file."""

    bucket: int
    map_name: str
    width: int
    height: int
    start: Position
    goal: Position
    reference_distance: float
    source_file: Path
    source_line: int


@dataclass(frozen=True)
class MapInstance:
    """A multi-agent problem created from one grid map and scenario file."""

    grid_map: GridMap
    scenario_file: Path
    starts: tuple[Position, ...]
    goals: tuple[Position, ...]
    tasks: tuple[ScenarioEntry, ...]

    @property
    def n_agents(self) -> int:
        return len(self.starts)

    def summary(self) -> str:
        return (
            f"Map: {self.grid_map.name}\n"
            f"Size: {self.grid_map.width} x {self.grid_map.height}\n"
            f"Free cells: {len(self.grid_map.free_cells):,}\n"
            f"Obstacles: {len(self.grid_map.obstacles):,}\n"
            f"Scenario: {self.scenario_file.name}\n"
            f"Agents: {self.n_agents}\n"
            f"Starts: {self.starts}\n"
            f"Goals: {self.goals}"
        )


def _read_nonempty_lines(path: Path) -> list[str]:
    try:
        return [
            line.rstrip("\n\r")
            for line in path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
    except OSError as exc:
        raise MapCreatorError(f"Could not read {path}: {exc}") from exc


def load_map_file(map_path: str | Path) -> GridMap:
    """
    Parse one benchmark .map file.

    The format header contains ``type octile`` even though this project uses
    only four-directional movement. The header check validates the file format;
    movement rules are defined later by GridMMDP and GridMap.neighbors4.
    """
    path = Path(map_path).resolve()

    if path.suffix.lower() != ".map":
        raise MapCreatorError(
            f"Expected a .map file, received: {path.name}"
        )

    if not path.is_file():
        raise MapCreatorError(
            f"Map file does not exist: {path}"
        )

    lines = _read_nonempty_lines(path)

    if len(lines) < 5:
        raise MapCreatorError(
            f"Map file is too short: {path}"
        )

    try:
        map_type_key, map_type = lines[0].split(maxsplit=1)
        height_key, height_text = lines[1].split(maxsplit=1)
        width_key, width_text = lines[2].split(maxsplit=1)
    except ValueError as exc:
        raise MapCreatorError(
            f"Malformed map header in {path}"
        ) from exc

    if map_type_key.lower() != "type":
        raise MapCreatorError(
            f"Expected 'type' header in {path}"
        )

    if map_type.lower() != "octile":
        raise MapCreatorError(
            f"Unsupported map type {map_type!r} in {path}; "
            "expected 'octile'"
        )

    if (
        height_key.lower() != "height"
        or width_key.lower() != "width"
    ):
        raise MapCreatorError(
            f"Expected height/width headers in {path}"
        )

    if lines[3].strip().lower() != "map":
        raise MapCreatorError(
            f"Expected 'map' marker on line 4 in {path}"
        )

    try:
        height = int(height_text)
        width = int(width_text)
    except ValueError as exc:
        raise MapCreatorError(
            f"Width and height must be integers in {path}"
        ) from exc

    if width <= 0 or height <= 0:
        raise MapCreatorError(
            f"Width and height must be positive in {path}"
        )

    grid_rows = lines[4:]

    if len(grid_rows) != height:
        raise MapCreatorError(
            f"{path.name}: header says height={height}, "
            f"but the file contains {len(grid_rows)} map rows"
        )

    obstacles: set[Position] = set()
    free_cells: set[Position] = set()

    for y, row in enumerate(grid_rows):
        if len(row) != width:
            raise MapCreatorError(
                f"{path.name}: row {y} has length {len(row)}, "
                f"expected {width}"
            )

        for x, symbol in enumerate(row):
            position = (x, y)

            if symbol in TRAVERSABLE_SYMBOLS:
                free_cells.add(position)
            elif symbol in BLOCKED_SYMBOLS:
                obstacles.add(position)
            else:
                raise MapCreatorError(
                    f"{path.name}: unknown map symbol {symbol!r} "
                    f"at {position}"
                )

    if not free_cells:
        raise MapCreatorError(
            f"{path.name} contains no traversable cells"
        )

    return GridMap(
        name=path.stem,
        path=path,
        width=width,
        height=height,
        grid=tuple(grid_rows),
        obstacles=frozenset(obstacles),
        free_cells=frozenset(free_cells),
    )


def load_scenario_file(
    scenario_path: str | Path,
    *,
    expected_map: GridMap | None = None,
) -> list[ScenarioEntry]:
    """
    Parse one .scen file.

    When ``expected_map`` is supplied, every task is checked against the map's
    filename, dimensions, boundaries, and blocked cells.
    """
    path = Path(scenario_path).resolve()

    if path.suffix.lower() != ".scen":
        raise MapCreatorError(
            f"Expected a .scen file, received: {path.name}"
        )

    if not path.is_file():
        raise MapCreatorError(
            f"Scenario file does not exist: {path}"
        )

    lines = _read_nonempty_lines(path)

    if not lines or not lines[0].lower().startswith("version"):
        raise MapCreatorError(
            f"Missing scenario version header in {path}"
        )

    entries: list[ScenarioEntry] = []

    for line_number, line in enumerate(lines[1:], start=2):
        fields = line.split()

        if len(fields) != 9:
            raise MapCreatorError(
                f"{path.name}, line {line_number}: expected 9 fields, "
                f"received {len(fields)}"
            )

        try:
            entry = ScenarioEntry(
                bucket=int(fields[0]),
                map_name=fields[1],
                width=int(fields[2]),
                height=int(fields[3]),
                start=(int(fields[4]), int(fields[5])),
                goal=(int(fields[6]), int(fields[7])),
                reference_distance=float(fields[8]),
                source_file=path,
                source_line=line_number,
            )
        except ValueError as exc:
            raise MapCreatorError(
                f"{path.name}, line {line_number}: invalid numeric value"
            ) from exc

        if expected_map is not None:
            _validate_scenario_entry(entry, expected_map)

        entries.append(entry)

    if not entries:
        raise MapCreatorError(
            f"No scenario tasks found in {path}"
        )

    return entries


def _validate_scenario_entry(
    entry: ScenarioEntry,
    grid_map: GridMap,
) -> None:
    expected_filename = grid_map.path.name

    if entry.map_name != expected_filename:
        raise MapCreatorError(
            f"{entry.source_file.name}, line {entry.source_line}: "
            f"scenario belongs to {entry.map_name!r}, but the loaded map is "
            f"{expected_filename!r}"
        )

    if (entry.width, entry.height) != (
        grid_map.width,
        grid_map.height,
    ):
        raise MapCreatorError(
            f"{entry.source_file.name}, line {entry.source_line}: "
            f"scenario dimensions are {entry.width}x{entry.height}, "
            f"but the map is {grid_map.width}x{grid_map.height}"
        )

    for label, position in (
        ("start", entry.start),
        ("goal", entry.goal),
    ):
        if not grid_map.in_bounds(position):
            raise MapCreatorError(
                f"{entry.source_file.name}, line {entry.source_line}: "
                f"{label} {position} is outside the map"
            )

        if not grid_map.is_free(position):
            raise MapCreatorError(
                f"{entry.source_file.name}, line {entry.source_line}: "
                f"{label} {position} is blocked"
            )


def discover_map_file(map_folder: str | Path) -> Path:
    """Find the single .map file directly inside a map folder."""
    folder = Path(map_folder).resolve()

    if not folder.is_dir():
        raise MapCreatorError(
            f"Map folder does not exist: {folder}"
        )

    map_files = sorted(folder.glob("*.map"))

    if not map_files:
        raise MapCreatorError(
            f"No .map file found directly inside {folder}"
        )

    if len(map_files) > 1:
        names = ", ".join(path.name for path in map_files)
        raise MapCreatorError(
            f"Expected one .map file in {folder}, found: {names}"
        )

    return map_files[0]


def discover_scenario_files(
    map_folder: str | Path,
) -> list[Path]:
    """Find .scen files recursively under the selected map folder."""
    folder = Path(map_folder).resolve()

    def natural_key(path: Path) -> list[object]:
        relative_path = str(path.relative_to(folder))
        return [
            int(part) if part.isdigit() else part.lower()
            for part in re.split(r"(\d+)", relative_path)
        ]

    files = sorted(
        folder.rglob("*.scen"),
        key=natural_key,
    )

    if not files:
        raise MapCreatorError(
            f"No .scen files found under {folder}"
        )

    return files


def _reachable_4way(
    grid_map: GridMap,
    start: Position,
    goal: Position,
) -> bool:
    """Check reachability under the project's four-directional movement."""
    if start == goal:
        return True

    queue: deque[Position] = deque([start])
    visited = {start}

    while queue:
        current = queue.popleft()

        for neighbor in grid_map.neighbors4(current):
            if neighbor == goal:
                return True

            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return False


def select_agent_tasks(
    entries: Sequence[ScenarioEntry],
    grid_map: GridMap,
    n_agents: int,
    *,
    offset: int = 0,
    require_unique_starts: bool = True,
    require_unique_goals: bool = True,
    require_4way_reachability: bool = True,
) -> tuple[ScenarioEntry, ...]:
    """
    Select a deterministic valid group of tasks for the requested agents.

    Invalid, duplicate, or unreachable entries are skipped until enough tasks
    have been collected.
    """
    if n_agents <= 0:
        raise MapCreatorError(
            "n_agents must be positive"
        )

    if offset < 0:
        raise MapCreatorError(
            "offset cannot be negative"
        )

    if offset >= len(entries):
        raise MapCreatorError(
            f"offset={offset} is outside the scenario file "
            f"with {len(entries)} entries"
        )

    selected: list[ScenarioEntry] = []
    used_starts: set[Position] = set()
    used_goals: set[Position] = set()

    for entry in entries[offset:]:
        _validate_scenario_entry(
            entry,
            grid_map,
        )

        if (
            require_unique_starts
            and entry.start in used_starts
        ):
            continue

        if (
            require_unique_goals
            and entry.goal in used_goals
        ):
            continue

        if (
            require_4way_reachability
            and not _reachable_4way(
                grid_map,
                entry.start,
                entry.goal,
            )
        ):
            continue

        selected.append(entry)
        used_starts.add(entry.start)
        used_goals.add(entry.goal)

        if len(selected) == n_agents:
            return tuple(selected)

    raise MapCreatorError(
        f"Could select only {len(selected)} valid tasks, "
        f"but {n_agents} agents were requested"
    )


def create_map_instance(
    map_folder: str | Path,
    n_agents: int,
    *,
    scenario_file: str | Path | None = None,
    scenario_number: int = 1,
    task_offset: int = 0,
    require_4way_reachability: bool = True,
) -> MapInstance:
    """Create one multi-agent instance from a map folder."""
    folder = Path(map_folder).resolve()
    map_path = discover_map_file(folder)
    grid_map = load_map_file(map_path)

    if scenario_file is None:
        scenario_files = discover_scenario_files(folder)

        if not 1 <= scenario_number <= len(scenario_files):
            raise MapCreatorError(
                f"scenario_number must be between 1 and "
                f"{len(scenario_files)}"
            )

        selected_scenario_path = scenario_files[
            scenario_number - 1
        ]
    else:
        scenario_path = Path(scenario_file)
        selected_scenario_path = (
            scenario_path
            if scenario_path.is_absolute()
            else folder / scenario_path
        ).resolve()

    entries = load_scenario_file(
        selected_scenario_path,
        expected_map=grid_map,
    )

    tasks = select_agent_tasks(
        entries,
        grid_map,
        n_agents,
        offset=task_offset,
        require_4way_reachability=require_4way_reachability,
    )

    return MapInstance(
        grid_map=grid_map,
        scenario_file=selected_scenario_path,
        starts=tuple(task.start for task in tasks),
        goals=tuple(task.goal for task in tasks),
        tasks=tasks,
    )


def render_instance(
    instance: MapInstance,
    output_path: str | Path,
) -> Path:
    """
    Save a visual validation image.

    If ``output_path`` is a directory, the function creates a default filename
    inside it, such as ``room-64-64-16_preview.png``.
    """
    try:
        import matplotlib.pyplot as plt
    except ImportError as exc:
        raise RuntimeError(
            "render_instance requires matplotlib. Install it with: "
            "python -m pip install matplotlib"
        ) from exc

    output = Path(output_path).resolve()

    if output.exists() and output.is_dir():
        output = output / (
            f"{instance.grid_map.name}_preview.png"
        )
    elif not output.suffix:
        output = output / (
            f"{instance.grid_map.name}_preview.png"
        )

    output.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    image = [
        [
            0
            if (x, y) in instance.grid_map.free_cells
            else 1
            for x in range(instance.grid_map.width)
        ]
        for y in range(instance.grid_map.height)
    ]

    figure, axis = plt.subplots(
        figsize=(10, 7)
    )

    axis.imshow(
        image,
        cmap="gray_r",
        interpolation="nearest",
    )

    start_x = [position[0] for position in instance.starts]
    start_y = [position[1] for position in instance.starts]
    goal_x = [position[0] for position in instance.goals]
    goal_y = [position[1] for position in instance.goals]

    axis.scatter(
        start_x,
        start_y,
        marker="o",
        label="Starts",
    )

    axis.scatter(
        goal_x,
        goal_y,
        marker="x",
        label="Goals",
    )

    for agent_index, (start, goal) in enumerate(
        zip(instance.starts, instance.goals),
        start=1,
    ):
        axis.text(
            start[0],
            start[1],
            str(agent_index),
            fontsize=8,
        )
        axis.text(
            goal[0],
            goal[1],
            str(agent_index),
            fontsize=8,
        )

    axis.set_title(
        f"{instance.grid_map.name}: {instance.n_agents} agents"
    )
    axis.set_xlim(
        -0.5,
        instance.grid_map.width - 0.5,
    )
    axis.set_ylim(
        instance.grid_map.height - 0.5,
        -0.5,
    )
    axis.set_aspect("equal")
    axis.legend()
    figure.tight_layout()
    figure.savefig(
        output,
        dpi=180,
    )
    plt.close(figure)

    return output


def _build_argument_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description=(
            "Create and validate a multi-agent instance from a map folder."
        )
    )
    parser.add_argument(
        "map_folder",
        type=Path,
        help="Folder containing one .map file and scenario files",
    )
    parser.add_argument(
        "--agents",
        type=int,
        default=2,
        help="Number of agent tasks to select (default: 2)",
    )
    parser.add_argument(
        "--scenario-number",
        type=int,
        default=1,
        help="One-based index of discovered .scen files (default: 1)",
    )
    parser.add_argument(
        "--task-offset",
        type=int,
        default=0,
        help="Skip this many scenario rows before selecting tasks",
    )
    parser.add_argument(
        "--preview",
        type=Path,
        help="Optional PNG path or output directory",
    )
    parser.add_argument(
        "--allow-non-four-way",
        action="store_true",
        help="Do not reject tasks unreachable with four-way movement",
    )
    return parser


def main() -> None:
    args = _build_argument_parser().parse_args()

    try:
        instance = create_map_instance(
            args.map_folder,
            args.agents,
            scenario_number=args.scenario_number,
            task_offset=args.task_offset,
            require_4way_reachability=(
                not args.allow_non_four_way
            ),
        )
    except MapCreatorError as exc:
        raise SystemExit(
            f"Map creation failed: {exc}"
        ) from exc

    print(instance.summary())

    if args.preview is not None:
        preview_path = render_instance(
            instance,
            args.preview,
        )
        print(
            f"Preview image saved to: {preview_path}"
        )


if __name__ == "__main__":
    main()



"""
Stochastic cooperative grid MMDP used by Baseline RTDP and OD-RTDP.

The module defines the environment only. It does not choose actions.

Objective
---------
The problem is formulated as a cost-minimization Stochastic Shortest Path:

    transition cost = number of agents that have not yet reached their goals

Summing this cost over a complete execution equals the sum of individual
arrival times when agents remain frozen after reaching their goals.

Stochastic movement
-------------------
Each intended movement succeeds with probability ``1 - slip`` and otherwise
leaves that agent in place. Outcomes are independent before collision handling.

Collision rule
--------------
When a raw joint outcome contains a vertex collision or an edge swap, the
complete joint movement is rejected and the environment remains in the current
state. No arbitrary numerical collision penalty is used.
"""

from collections import OrderedDict, defaultdict
from contextlib import contextmanager
from dataclasses import dataclass
from itertools import product
import math
import random
from typing import Iterator



State = tuple[Position, ...]
Action = str
JointAction = tuple[Action, ...]
Transition = tuple[State, float]


ACTIONS: tuple[Action, ...] = (
    "stay",
    "up",
    "down",
    "left",
    "right",
)


MOVE: dict[Action, Position] = {
    "stay": (0, 0),
    "up": (0, -1),
    "down": (0, 1),
    "left": (-1, 0),
    "right": (1, 0),
}


@dataclass(frozen=True)
class MMDPConfig:
    """Parameters defining the stochastic grid environment.

    ``transition_cache_max_entries`` bounds each of the raw and resolved LRU
    transition caches.  ``None`` keeps the old unbounded behavior and ``0``
    disables new cache entries.  Bounding the caches is important on large
    multi-agent instances, where the number of state-action pairs grows as
    5**n and an unbounded cache can dominate memory.
    """

    slip_to_stay_probability: float = 0.20
    freeze_agents_at_goal: bool = True
    reject_conflicting_transitions: bool = True
    transition_cache_max_entries: int | None = None

    def __post_init__(self) -> None:
        if not 0.0 <= self.slip_to_stay_probability <= 1.0:
            raise ValueError(
                "slip_to_stay_probability must be between 0 and 1"
            )

        if (
            self.transition_cache_max_entries is not None
            and self.transition_cache_max_entries < 0
        ):
            raise ValueError(
                "transition_cache_max_entries must be non-negative or None"
            )


class GridMMDP:
    """Centralized cooperative MMDP built from a MapInstance."""

    def __init__(
        self,
        instance: MapInstance,
        config: MMDPConfig | None = None,
    ) -> None:
        self.instance = instance
        self.config = config or MMDPConfig()

        # Transition distributions are stationary for a fixed MMDP.
        # Cache both raw and collision-resolved distributions so repeated
        # Bellman backups do not rebuild the same Cartesian products.
        self._raw_transition_cache: OrderedDict[
            tuple[State, JointAction],
            tuple[Transition, ...],
        ] = OrderedDict()
        self._resolved_transition_cache: OrderedDict[
            tuple[State, JointAction],
            tuple[Transition, ...],
        ] = OrderedDict()

        # Planning benefits from full transition memoization. During policy
        # evaluation, however, evaluating all candidate actions for a new
        # state can insert 5**n distributions even though only one action is
        # executed. The write switch lets evaluation reuse existing planning
        # entries without filling the cache with every rejected candidate.
        self._transition_cache_write_enabled = True

        self._raw_transition_cache_hits = 0
        self._raw_transition_cache_misses = 0
        self._raw_transition_cache_writes = 0
        self._raw_transition_cache_evictions = 0
        self._resolved_transition_cache_hits = 0
        self._resolved_transition_cache_misses = 0
        self._resolved_transition_cache_writes = 0
        self._resolved_transition_cache_evictions = 0

        self._validate_instance()

    @property
    def map_name(self) -> str:
        return self.instance.grid_map.name

    @property
    def width(self) -> int:
        return self.instance.grid_map.width

    @property
    def height(self) -> int:
        return self.instance.grid_map.height

    @property
    def obstacles(self) -> frozenset[Position]:
        return self.instance.grid_map.obstacles

    @property
    def free_cells(self) -> frozenset[Position]:
        return self.instance.grid_map.free_cells

    @property
    def starts(self) -> State:
        return self.instance.starts

    @property
    def goals(self) -> State:
        return self.instance.goals

    @property
    def n_agents(self) -> int:
        return self.instance.n_agents

    def _validate_instance(self) -> None:
        if self.n_agents <= 0:
            raise ValueError(
                "MapInstance must contain at least one agent"
            )

        if len(self.starts) != len(self.goals):
            raise ValueError(
                "MapInstance must contain the same number of starts and goals"
            )

        if len(set(self.starts)) != len(self.starts):
            raise ValueError(
                "Agent start positions must be unique"
            )

        if len(set(self.goals)) != len(self.goals):
            raise ValueError(
                "Agent goal positions must be unique"
            )

        for label, positions in (
            ("start", self.starts),
            ("goal", self.goals),
        ):
            for agent_index, position in enumerate(positions):
                if position not in self.free_cells:
                    raise ValueError(
                        f"Agent {agent_index} {label} "
                        f"{position} is not a free cell"
                    )

    def initial_state(self) -> State:
        return self.starts

    def goal_state(self) -> State:
        return self.goals

    def validate_state(self, state: State) -> None:
        if len(state) != self.n_agents:
            raise ValueError(
                f"State has {len(state)} positions; "
                f"expected {self.n_agents}"
            )

        for agent_index, position in enumerate(state):
            if position not in self.free_cells:
                raise ValueError(
                    f"Agent {agent_index} is on an invalid cell: {position}"
                )

    def is_terminal(self, state: State) -> bool:
        self.validate_state(state)
        return state == self.goals

    def validate_joint_action(
        self,
        joint_action: JointAction,
    ) -> None:
        if len(joint_action) != self.n_agents:
            raise ValueError(
                f"Joint action has {len(joint_action)} actions; "
                f"expected {self.n_agents}"
            )

        invalid_actions = [
            action
            for action in joint_action
            if action not in MOVE
        ]

        if invalid_actions:
            raise ValueError(
                f"Unknown actions: {invalid_actions}"
            )

    def unfinished_agent_count(
        self,
        state: State,
    ) -> int:
        """Return the number of agents not yet at their assigned goals."""
        self.validate_state(state)

        return sum(
            position != goal
            for position, goal in zip(state, self.goals)
        )

    def transition_cost(
        self,
        state: State,
        joint_action: JointAction,
        next_state: State,
    ) -> float:
        """
        Return the non-negative cost of one real environment transition.

        The current model uses:

            c(s,a,s') = number of unfinished agents in s

        The action and successor are validated even though the numerical value
        currently depends only on the current state.
        """
        self.validate_state(state)
        self.validate_joint_action(joint_action)
        self.validate_state(next_state)

        if self.is_terminal(state):
            return 0.0

        return float(
            self.unfinished_agent_count(state)
        )

    def move_one(
        self,
        agent_index: int,
        position: Position,
        action: Action,
    ) -> Position:
        """Apply one intended deterministic movement."""
        if not 0 <= agent_index < self.n_agents:
            raise IndexError(
                f"Agent index {agent_index} is outside the valid range"
            )

        if action not in MOVE:
            raise ValueError(
                f"Unknown action: {action!r}"
            )

        if position not in self.free_cells:
            raise ValueError(
                f"Position {position} is not a free cell"
            )

        if (
            self.config.freeze_agents_at_goal
            and position == self.goals[agent_index]
        ):
            return position

        dx, dy = MOVE[action]
        candidate = (
            position[0] + dx,
            position[1] + dy,
        )

        if candidate in self.free_cells:
            return candidate

        return position

    def individual_outcomes(
        self,
        agent_index: int,
        position: Position,
        action: Action,
    ) -> tuple[tuple[Position, float], ...]:
        """Return all possible stochastic outcomes for one agent."""
        intended_position = self.move_one(
            agent_index,
            position,
            action,
        )

        success_probability = (
            1.0 - self.config.slip_to_stay_probability
        )

        outcomes: dict[Position, float] = defaultdict(float)
        outcomes[intended_position] += success_probability
        outcomes[position] += self.config.slip_to_stay_probability

        result = tuple(
            (next_position, probability)
            for next_position, probability in outcomes.items()
            if probability > 0.0
        )

        probability_sum = sum(
            probability
            for _, probability in result
        )

        if not math.isclose(
            probability_sum,
            1.0,
            rel_tol=0.0,
            abs_tol=1e-12,
        ):
            raise RuntimeError(
                "Individual transition probabilities sum to "
                f"{probability_sum}, not 1"
            )

        return result

    def _cache_lookup(
        self,
        cache: OrderedDict[
            tuple[State, JointAction],
            tuple[Transition, ...],
        ],
        key: tuple[State, JointAction],
    ) -> tuple[Transition, ...] | None:
        """Return and mark an LRU cache entry as recently used."""
        try:
            value = cache[key]
        except KeyError:
            return None
        cache.move_to_end(key)
        return value

    def _cache_store(
        self,
        cache: OrderedDict[
            tuple[State, JointAction],
            tuple[Transition, ...],
        ],
        key: tuple[State, JointAction],
        value: tuple[Transition, ...],
        *,
        raw: bool,
    ) -> bool:
        """Store one transition distribution and enforce the LRU bound."""
        if not self._transition_cache_write_enabled:
            return False

        limit = self.config.transition_cache_max_entries
        if limit == 0:
            return False

        cache[key] = value
        cache.move_to_end(key)

        if limit is not None and len(cache) > limit:
            cache.popitem(last=False)
            if raw:
                self._raw_transition_cache_evictions += 1
            else:
                self._resolved_transition_cache_evictions += 1

        return True

    def _raw_joint_transitions(
        self,
        state: State,
        joint_action: JointAction,
    ) -> tuple[Transition, ...]:
        """
        Return joint outcomes before applying the collision rejection rule.

        The distribution is cached because it depends only on the immutable
        MMDP definition, the current state, and the joint action.
        """
        self.validate_state(state)
        self.validate_joint_action(joint_action)

        cache_key = (
            state,
            joint_action,
        )

        cached = self._cache_lookup(
            self._raw_transition_cache,
            cache_key,
        )

        if cached is not None:
            self._raw_transition_cache_hits += 1
            return cached

        self._raw_transition_cache_misses += 1

        if self.is_terminal(state):
            terminal_transition = ((state, 1.0),)
            if self._cache_store(
                self._raw_transition_cache,
                cache_key,
                terminal_transition,
                raw=True,
            ):
                self._raw_transition_cache_writes += 1
            return terminal_transition

        per_agent_outcomes = [
            self.individual_outcomes(
                agent_index,
                position,
                action,
            )
            for agent_index, (position, action)
            in enumerate(zip(state, joint_action))
        ]

        merged: dict[State, float] = defaultdict(float)

        for combination in product(*per_agent_outcomes):
            next_state: State = tuple(
                position
                for position, _ in combination
            )

            probability = math.prod(
                outcome_probability
                for _, outcome_probability in combination
            )

            merged[next_state] += probability

        transitions = tuple(
            (next_state, probability)
            for next_state, probability in merged.items()
            if probability > 0.0
        )

        self._validate_probability_sum(
            transitions,
            label="Raw joint",
        )

        if self._cache_store(
            self._raw_transition_cache,
            cache_key,
            transitions,
            raw=True,
        ):
            self._raw_transition_cache_writes += 1

        return transitions

    @staticmethod
    def vertex_conflict_count(
        state: State,
    ) -> int:
        """Count pairs of agents occupying the same cell."""
        counts: dict[Position, int] = defaultdict(int)

        for position in state:
            counts[position] += 1

        return sum(
            count * (count - 1) // 2
            for count in counts.values()
            if count > 1
        )

    @staticmethod
    def edge_swap_conflict_count(
        state: State,
        next_state: State,
    ) -> int:
        """Count pairs of agents that swap positions in one step."""
        if len(state) != len(next_state):
            raise ValueError(
                "state and next_state must contain the same number of agents"
            )

        conflicts = 0

        for first_agent in range(len(state)):
            for second_agent in range(
                first_agent + 1,
                len(state),
            ):
                if (
                    state[first_agent] == next_state[second_agent]
                    and state[second_agent] == next_state[first_agent]
                    and state[first_agent] != state[second_agent]
                ):
                    conflicts += 1

        return conflicts

    def has_conflict(
        self,
        state: State,
        next_state: State,
    ) -> bool:
        """Return True for a vertex conflict or an edge-swap conflict."""
        return (
            self.vertex_conflict_count(next_state) > 0
            or self.edge_swap_conflict_count(state, next_state) > 0
        )

    def conflict_probability(
        self,
        state: State,
        joint_action: JointAction,
    ) -> float:
        """Return the probability mass of conflicting raw outcomes."""
        return sum(
            probability
            for next_state, probability
            in self._raw_joint_transitions(state, joint_action)
            if self.has_conflict(state, next_state)
        )

    def self_loop_probability(
        self,
        state: State,
        joint_action: JointAction,
    ) -> float:
        """Return ``P(next_state == state | state, joint_action)``.

        This includes every way a real step can leave the full joint state
        unchanged: independent slips, explicit or blocked ``stay`` actions,
        and rejection of a conflicting raw outcome.  It is therefore more
        informative for policy tie breaking than collision probability alone.
        """
        self.validate_state(state)
        self.validate_joint_action(joint_action)

        return sum(
            probability
            for next_state, probability in self.joint_transitions(
                state,
                joint_action,
            )
            if next_state == state
        )


    def action_risk_breakdown(
        self,
        state: State,
        joint_action: JointAction,
    ) -> dict[str, float | int]:
        """Return an interpretable breakdown of one joint action.

        Probabilities are computed from raw stochastic outcomes, before the
        collision-rejection rule merges them into the current state.  The
        categories are mutually interpretable but not all are mutually
        exclusive: ``self_loop_probability`` is the final resolved probability
        of staying in the same joint state, while vertex/edge probabilities
        identify the collision component and ``noncollision_no_motion``
        identifies raw outcomes that already equal the current state.
        """
        self.validate_state(state)
        self.validate_joint_action(joint_action)

        vertex_probability = 0.0
        edge_swap_probability = 0.0
        any_conflict_probability = 0.0
        noncollision_no_motion_probability = 0.0

        for raw_next_state, probability in self._raw_joint_transitions(
            state, joint_action
        ):
            vertex = self.vertex_conflict_count(raw_next_state) > 0
            edge = self.edge_swap_conflict_count(state, raw_next_state) > 0
            if vertex:
                vertex_probability += probability
            if edge:
                edge_swap_probability += probability
            if vertex or edge:
                any_conflict_probability += probability
            elif raw_next_state == state:
                noncollision_no_motion_probability += probability

        unfinished_stay_actions = 0
        unfinished_blocked_actions = 0
        movable_unfinished_actions = 0
        frozen_agents = 0

        for index, (position, action, goal) in enumerate(
            zip(state, joint_action, self.goals)
        ):
            if position == goal and self.config.freeze_agents_at_goal:
                frozen_agents += 1
                continue
            if action == "stay":
                unfinished_stay_actions += 1
                continue
            intended = self.move_one(index, position, action)
            if intended == position:
                unfinished_blocked_actions += 1
            else:
                movable_unfinished_actions += 1

        return {
            "self_loop_probability": self.self_loop_probability(
                state, joint_action
            ),
            "conflict_probability": any_conflict_probability,
            "vertex_conflict_probability": vertex_probability,
            "edge_swap_probability": edge_swap_probability,
            "noncollision_no_motion_probability": (
                noncollision_no_motion_probability
            ),
            "unfinished_stay_actions": unfinished_stay_actions,
            "unfinished_blocked_actions": unfinished_blocked_actions,
            "movable_unfinished_actions": movable_unfinished_actions,
            "frozen_agents": frozen_agents,
        }

    def joint_transitions(
        self,
        state: State,
        joint_action: JointAction,
    ) -> tuple[Transition, ...]:
        """
        Return every legal resolved successor and its probability.

        When collision rejection is enabled, each conflicting raw outcome is
        converted into a transition back to the current state. Duplicate
        successors are then merged. The resolved distribution is memoized.
        """
        self.validate_state(state)
        self.validate_joint_action(joint_action)

        cache_key = (
            state,
            joint_action,
        )

        cached = self._cache_lookup(
            self._resolved_transition_cache,
            cache_key,
        )

        if cached is not None:
            self._resolved_transition_cache_hits += 1
            return cached

        self._resolved_transition_cache_misses += 1

        raw_transitions = self._raw_joint_transitions(
            state,
            joint_action,
        )

        if not self.config.reject_conflicting_transitions:
            if self._cache_store(
                self._resolved_transition_cache,
                cache_key,
                raw_transitions,
                raw=False,
            ):
                self._resolved_transition_cache_writes += 1
            return raw_transitions

        merged: dict[State, float] = defaultdict(float)

        for raw_next_state, probability in raw_transitions:
            resolved_state = (
                state
                if self.has_conflict(state, raw_next_state)
                else raw_next_state
            )

            merged[resolved_state] += probability

        transitions = tuple(
            (next_state, probability)
            for next_state, probability in merged.items()
            if probability > 0.0
        )

        self._validate_probability_sum(
            transitions,
            label="Resolved joint",
        )

        if self._cache_store(
            self._resolved_transition_cache,
            cache_key,
            transitions,
            raw=False,
        ):
            self._resolved_transition_cache_writes += 1

        return transitions

    @contextmanager
    def transition_cache_writes(
        self,
        enabled: bool,
    ) -> Iterator[None]:
        """Temporarily enable or suppress new transition-cache writes.

        Cache reads remain active. Nested contexts are conservative: an inner
        context cannot re-enable writes while an outer context disabled them.
        This is used during greedy policy extraction in evaluation so only the
        action actually executed is cached afterwards.
        """
        previous = self._transition_cache_write_enabled
        self._transition_cache_write_enabled = previous and enabled
        try:
            yield
        finally:
            self._transition_cache_write_enabled = previous

    def reset_transition_cache_stats(self) -> None:
        """Reset hit/miss/write counters without clearing distributions."""
        self._raw_transition_cache_hits = 0
        self._raw_transition_cache_misses = 0
        self._raw_transition_cache_writes = 0
        self._raw_transition_cache_evictions = 0
        self._resolved_transition_cache_hits = 0
        self._resolved_transition_cache_misses = 0
        self._resolved_transition_cache_writes = 0
        self._resolved_transition_cache_evictions = 0

    def transition_cache_stats(self) -> dict[str, int | bool | None]:
        """Return cache sizes and counters for performance diagnostics."""
        return {
            "write_enabled": self._transition_cache_write_enabled,
            "max_entries_per_cache": self.config.transition_cache_max_entries,
            "raw_entries": len(self._raw_transition_cache),
            "resolved_entries": len(self._resolved_transition_cache),
            "raw_hits": self._raw_transition_cache_hits,
            "raw_misses": self._raw_transition_cache_misses,
            "raw_writes": self._raw_transition_cache_writes,
            "raw_evictions": self._raw_transition_cache_evictions,
            "resolved_hits": self._resolved_transition_cache_hits,
            "resolved_misses": self._resolved_transition_cache_misses,
            "resolved_writes": self._resolved_transition_cache_writes,
            "resolved_evictions": self._resolved_transition_cache_evictions,
        }

    def clear_transition_cache(self) -> None:
        """
        Clear memoized transition distributions.

        This is normally unnecessary because one GridMMDP object represents an
        immutable environment configuration. It is provided for diagnostics
        and memory-control experiments.
        """
        self._raw_transition_cache.clear()
        self._resolved_transition_cache.clear()

    @staticmethod
    def _validate_probability_sum(
        transitions: tuple[Transition, ...],
        *,
        label: str,
    ) -> None:
        probability_sum = sum(
            probability
            for _, probability in transitions
        )

        if not math.isclose(
            probability_sum,
            1.0,
            rel_tol=0.0,
            abs_tol=1e-10,
        ):
            raise RuntimeError(
                f"{label} transition probabilities sum to "
                f"{probability_sum}, not 1"
            )

    @staticmethod
    def sample_from_transitions(
        transitions: tuple[Transition, ...],
        rng: random.Random,
    ) -> State:
        """Sample from a transition tuple that was already computed."""
        if not transitions:
            raise ValueError("transitions cannot be empty")

        roll = rng.random()
        cumulative_probability = 0.0

        for next_state, probability in transitions:
            cumulative_probability += probability
            if roll <= cumulative_probability:
                return next_state

        return transitions[-1][0]

    def sample_next(
        self,
        state: State,
        joint_action: JointAction,
        rng: random.Random,
    ) -> State:
        """Sample one actual successor after an action has been selected."""
        transitions = self.joint_transitions(state, joint_action)
        return self.sample_from_transitions(transitions, rng)

    def all_joint_actions(
        self,
    ) -> Iterator[JointAction]:
        """Generate all complete joint actions lazily."""
        return product(
            ACTIONS,
            repeat=self.n_agents,
        )

    def summary(self) -> str:
        return (
            f"MMDP map: {self.map_name}\n"
            f"Size: {self.width} x {self.height}\n"
            f"Agents: {self.n_agents}\n"
            f"Scenario: {self.instance.scenario_file.name}\n"
            f"Slip-to-stay probability: "
            f"{self.config.slip_to_stay_probability:.2f}\n"
            f"Freeze agents at goal: "
            f"{self.config.freeze_agents_at_goal}\n"
            f"Reject conflicting transitions: "
            f"{self.config.reject_conflicting_transitions}\n"
            f"Transition cache max entries per cache: "
            f"{self.config.transition_cache_max_entries}\n"
            "Transition cost: number of unfinished agents"
        )



"""
Slip-aware shortest-path heuristics and structured tie guidance.

The numeric heuristic remains an admissible lower bound for the cooperative
Stochastic Shortest Path objective.

For one isolated agent at shortest-path distance d, with movement success
probability q = 1 - slip, the expected remaining number of charged steps is:

    d / q

because every successful unit of shortest-path progress requires a geometric
number of attempts with mean 1 / q.

Baseline RTDP therefore uses:

    h(state) = sum_i d_i(state) / q

while ignoring collisions and inter-agent waiting. Ignoring those interactions
can only make the estimate smaller than the real cooperative cost.

For an OD state (state, prefix), actions already fixed in the prefix are
accounted for using a collision-safe one-agent lower bound.  A fixed action's
isolated expected future value is capped by the agent's current value:

    min(H_i(current), E[H_i(after isolated action)])

This cap is necessary because collision rejection can replace a harmful raw
move by the current state.  Without the cap, a prefix that fixes a move away
from the goal can be overestimated and cease to be admissible.  Agents that
have not yet received an action are assumed to choose their individually best
safe one-step action.  The empty-prefix OD heuristic remains exactly equal to
the Baseline heuristic.

Shortest-path distance alone deliberately treats every equally short route as
identical. To avoid large arbitrary tie sets, this module also exposes
structured secondary guidance keys. They are used only after Bellman/Q values
are tied, so they do not replace or perturb the primary optimization objective.
The guidance prefers, in order:

- lower probability of remaining in exactly the same joint state;
- lower collision risk;
- fewer deterministic/partial conflicts;
- fewer moves into another agent's goal or current cell;
- genuine shortest-path progress over waiting or blocked moves;
- branches with more remaining shortest paths;
- positions with more future shortest-path choices and local mobility.
"""

from collections import deque
from dataclasses import dataclass, field
import math
from typing import Sequence

    ACTIONS,
    Action,
    GridMMDP,
    JointAction,
    State,
)


DistanceTable = dict[Position, int]
LogPathCountTable = dict[Position, float]
GuidanceKey = tuple[float | int, ...]


def build_distance_table(
    mdp: GridMMDP,
    goal: Position,
) -> DistanceTable:
    """Compute exact four-directional shortest-path distances to one goal."""
    if goal not in mdp.free_cells:
        raise ValueError(
            f"Cannot build a distance table for blocked goal {goal}"
        )

    distances: DistanceTable = {goal: 0}
    queue: deque[Position] = deque([goal])

    while queue:
        current = queue.popleft()
        next_distance = distances[current] + 1

        for neighbor in mdp.instance.grid_map.neighbors4(current):
            if neighbor in distances:
                continue

            distances[neighbor] = next_distance
            queue.append(neighbor)

    return distances


def _logsumexp(values: list[float]) -> float:
    """Stable logarithm of a sum of exponentials."""
    if not values:
        return -math.inf

    maximum = max(values)
    return maximum + math.log(
        sum(math.exp(value - maximum) for value in values)
    )


def build_shortest_path_log_count_table(
    mdp: GridMMDP,
    distances: DistanceTable,
    goal: Position,
) -> LogPathCountTable:
    """
    Store log(number of shortest paths) from every reachable cell to the goal.

    Using logarithms avoids enormous integers on large open maps. The value is
    used only as secondary guidance: a branch that preserves more shortest
    routes is preferred when the Bellman values are tied.
    """
    log_counts: LogPathCountTable = {goal: 0.0}  # log(1)

    for position, distance in sorted(
        distances.items(),
        key=lambda item: item[1],
    ):
        if distance == 0:
            continue

        predecessor_logs = [
            log_counts[neighbor]
            for neighbor in mdp.instance.grid_map.neighbors4(position)
            if distances.get(neighbor) == distance - 1
        ]

        if not predecessor_logs:
            raise RuntimeError(
                f"Reachable cell {position} has no shortest-path predecessor"
            )

        log_counts[position] = _logsumexp(predecessor_logs)

    return log_counts


@dataclass
class ShortestPathHeuristic:
    """Obstacle-aware, slip-aware shortest-path lower bound and tie guidance."""

    mdp: GridMMDP

    distance_tables: tuple[DistanceTable, ...] = field(
        init=False,
        repr=False,
    )
    shortest_path_log_tables: tuple[LogPathCountTable, ...] = field(
        init=False,
        repr=False,
    )
    movement_success_probability: float = field(
        init=False,
    )

    def __post_init__(self) -> None:
        self.movement_success_probability = (
            1.0 - self.mdp.config.slip_to_stay_probability
        )

        if self.movement_success_probability <= 0.0:
            raise ValueError(
                "ShortestPathHeuristic requires a positive movement success "
                "probability"
            )

        self.distance_tables = tuple(
            build_distance_table(self.mdp, goal)
            for goal in self.mdp.goals
        )

        self.shortest_path_log_tables = tuple(
            build_shortest_path_log_count_table(
                self.mdp,
                distances,
                goal,
            )
            for distances, goal in zip(
                self.distance_tables,
                self.mdp.goals,
            )
        )

        self._validate_initial_positions()

    def _validate_initial_positions(self) -> None:
        for agent_index, start in enumerate(self.mdp.starts):
            if start not in self.distance_tables[agent_index]:
                raise ValueError(
                    f"Agent {agent_index} cannot reach its goal "
                    f"{self.mdp.goals[agent_index]} from start {start}"
                )

    def agent_distance(
        self,
        agent_index: int,
        position: Position,
    ) -> float:
        """Return one agent's deterministic shortest-path distance."""
        if not 0 <= agent_index < self.mdp.n_agents:
            raise IndexError(
                f"Agent index {agent_index} is outside the valid range"
            )

        if position not in self.mdp.free_cells:
            raise ValueError(
                f"Position {position} is not a free map cell"
            )

        return float(
            self.distance_tables[agent_index].get(
                position,
                math.inf,
            )
        )

    def stochastic_agent_distance(
        self,
        agent_index: int,
        position: Position,
    ) -> float:
        """Return the isolated-agent slip-aware lower bound d / q."""
        distance = self.agent_distance(agent_index, position)

        if math.isinf(distance):
            return math.inf

        return distance / self.movement_success_probability

    def __call__(
        self,
        state: State,
    ) -> float:
        """
        Return the Baseline lower bound:

            h(state) = sum_i shortest_distance_i / movement_success_probability
        """
        self.mdp.validate_state(state)

        total = 0.0

        for agent_index, position in enumerate(state):
            contribution = self.stochastic_agent_distance(
                agent_index,
                position,
            )

            if math.isinf(contribution):
                return math.inf

            total += contribution

        return total

    def deterministic_result(
        self,
        agent_index: int,
        position: Position,
        action: Action,
    ) -> Position:
        """Return the intended result before slip is sampled."""
        return self.mdp.move_one(
            agent_index,
            position,
            action,
        )

    def distance_after_action(
        self,
        agent_index: int,
        position: Position,
        action: Action,
    ) -> float:
        """Return deterministic shortest-path distance after one intended move."""
        next_position = self.deterministic_result(
            agent_index,
            position,
            action,
        )

        return self.agent_distance(
            agent_index,
            next_position,
        )

    def expected_stochastic_distance_after_action(
        self,
        agent_index: int,
        position: Position,
        action: Action,
    ) -> float:
        """
        Expected isolated-agent lower bound after executing one stochastic move.

        The immediate real-step cost is not included here; od_value adds it once
        for every unfinished agent before summing these future contributions.
        """
        expected = 0.0

        for next_position, probability in self.mdp.individual_outcomes(
            agent_index,
            position,
            action,
        ):
            future = self.stochastic_agent_distance(
                agent_index,
                next_position,
            )

            if math.isinf(future):
                return math.inf

            expected += probability * future

        return expected

    def best_optimistic_distance(
        self,
        agent_index: int,
        position: Position,
    ) -> float:
        """Backward-compatible deterministic one-step distance helper."""
        return min(
            self.distance_after_action(
                agent_index,
                position,
                action,
            )
            for action in ACTIONS
        )

    def safe_expected_stochastic_distance_after_action(
        self,
        agent_index: int,
        position: Position,
        action: Action,
    ) -> float:
        """Return a collision-safe future lower bound for one fixed action.

        Collision rejection maps a conflicting raw outcome back to the current
        joint state.  For a move that increases an agent's isolated distance,
        the raw one-agent expectation can therefore be larger than the true
        cooperative successor value.  Capping it by the current isolated value
        preserves admissibility for every possible collision pattern.
        """
        current = self.stochastic_agent_distance(
            agent_index,
            position,
        )
        expected = self.expected_stochastic_distance_after_action(
            agent_index,
            position,
            action,
        )
        return min(current, expected)

    def best_expected_stochastic_distance(
        self,
        agent_index: int,
        position: Position,
    ) -> float:
        """Best collision-safe isolated future lower bound after one action."""
        return min(
            self.safe_expected_stochastic_distance_after_action(
                agent_index,
                position,
                action,
            )
            for action in ACTIONS
        )

    def shortest_path_log_count(
        self,
        agent_index: int,
        position: Position,
    ) -> float:
        """Return log(number of shortest routes) from position to the goal."""
        return self.shortest_path_log_tables[agent_index].get(
            position,
            -math.inf,
        )

    def progress_action_count(
        self,
        agent_index: int,
        position: Position,
    ) -> int:
        """Count actions that reduce deterministic shortest-path distance by 1."""
        current_distance = self.agent_distance(
            agent_index,
            position,
        )

        if current_distance <= 0.0 or math.isinf(current_distance):
            return 0

        return sum(
            self.distance_after_action(
                agent_index,
                position,
                action,
            )
            == current_distance - 1.0
            for action in ACTIONS
        )

    def local_degree(
        self,
        position: Position,
    ) -> int:
        """Return the number of legal four-neighbor movements from a cell."""
        return sum(
            1
            for _ in self.mdp.instance.grid_map.neighbors4(position)
        )

    def individual_stay_probability(
        self,
        agent_index: int,
        position: Position,
        action: Action,
    ) -> float:
        """Return the chance that one agent remains on its current cell."""
        return sum(
            probability
            for next_position, probability in self.mdp.individual_outcomes(
                agent_index,
                position,
                action,
            )
            if next_position == position
        )

    def minimum_individual_stay_probability(
        self,
        agent_index: int,
        position: Position,
    ) -> float:
        """Best isolated-agent chance of leaving the current cell."""
        return min(
            self.individual_stay_probability(
                agent_index,
                position,
                action,
            )
            for action in ACTIONS
        )

    def optimistic_prefix_self_loop_probability(
        self,
        state: State,
        prefix: Sequence[Action],
    ) -> float:
        """Optimistic all-agents-stay probability for an incomplete prefix.

        Fixed prefix actions use their exact isolated stay probabilities.
        Every unselected agent is assumed to choose the action with the lowest
        isolated stay probability.  Collision rejection is deliberately not
        added here; partial collision indicators remain separate guidance
        fields.  The value is used only to order Bellman-tied OD operators.
        """
        self.mdp.validate_state(state)

        if len(prefix) > self.mdp.n_agents:
            raise ValueError(
                f"Prefix contains {len(prefix)} actions, "
                f"but the problem has only {self.mdp.n_agents} agents"
            )

        probability = 1.0

        for agent_index, position in enumerate(state):
            if agent_index < len(prefix):
                stay_probability = self.individual_stay_probability(
                    agent_index,
                    position,
                    prefix[agent_index],
                )
            else:
                stay_probability = self.minimum_individual_stay_probability(
                    agent_index,
                    position,
                )

            probability *= stay_probability

        return probability

    def od_value(
        self,
        state: State,
        prefix: Sequence[Action],
    ) -> float:
        """
        Return the slip-aware lower bound for an OD state.

        A future real transition charges one unit per unfinished agent. For a
        fixed prefix action we add a collision-safe future lower bound:

            min(current isolated value, isolated expected value after action)

        Unselected agents receive their best safe individual one-step action.
        This remains optimistic even when a harmful raw move would be rejected
        by a collision and converted back to the current state.
        """
        self.mdp.validate_state(state)

        if len(prefix) > self.mdp.n_agents:
            raise ValueError(
                f"Prefix contains {len(prefix)} actions, "
                f"but the problem has only {self.mdp.n_agents} agents"
            )

        invalid_actions = [
            action
            for action in prefix
            if action not in ACTIONS
        ]

        if invalid_actions:
            raise ValueError(
                f"Prefix contains unknown actions: {invalid_actions}"
            )

        if self.mdp.is_terminal(state):
            return 0.0

        estimate = float(
            self.mdp.unfinished_agent_count(state)
        )

        for agent_index, position in enumerate(state):
            if position == self.mdp.goals[agent_index]:
                continue

            if agent_index < len(prefix):
                future = self.safe_expected_stochastic_distance_after_action(
                    agent_index,
                    position,
                    prefix[agent_index],
                )
            else:
                future = self.best_expected_stochastic_distance(
                    agent_index,
                    position,
                )

            if math.isinf(future):
                return math.inf

            estimate += future

        return estimate

    def _intended_joint_state(
        self,
        state: State,
        joint_action: JointAction,
    ) -> State:
        return tuple(
            self.deterministic_result(
                agent_index,
                position,
                action,
            )
            for agent_index, (position, action) in enumerate(
                zip(state, joint_action)
            )
        )

    def _foreign_goal_occupancy_count(
        self,
        intended_state: State,
    ) -> int:
        count = 0

        for agent_index, position in enumerate(intended_state):
            for other_index, other_goal in enumerate(self.mdp.goals):
                if other_index != agent_index and position == other_goal:
                    count += 1

        return count

    def joint_action_guidance_key(
        self,
        state: State,
        joint_action: JointAction,
    ) -> GuidanceKey:
        """
        Lexicographic secondary key for Q-tied complete joint actions.

        Smaller tuples are preferred. This method never changes the primary
        Q-value and therefore only resolves genuine numerical ties.
        """
        self.mdp.validate_state(state)
        self.mdp.validate_joint_action(joint_action)

        intended_state = self._intended_joint_state(
            state,
            joint_action,
        )

        self_loop_probability = self.mdp.self_loop_probability(
            state,
            joint_action,
        )

        conflict_probability = self.mdp.conflict_probability(
            state,
            joint_action,
        )

        deterministic_conflicts = (
            self.mdp.vertex_conflict_count(intended_state)
            + self.mdp.edge_swap_conflict_count(state, intended_state)
        )

        foreign_goal_occupancy = self._foreign_goal_occupancy_count(
            intended_state
        )

        nonprogress_count = 0
        stationary_count = 0
        total_log_path_count = 0.0
        total_progress_options = 0
        total_degree = 0

        for agent_index, (position, next_position) in enumerate(
            zip(state, intended_state)
        ):
            if position == self.mdp.goals[agent_index]:
                continue

            current_distance = self.agent_distance(
                agent_index,
                position,
            )
            next_distance = self.agent_distance(
                agent_index,
                next_position,
            )

            if next_distance >= current_distance:
                nonprogress_count += 1

            if next_position == position:
                stationary_count += 1

            total_log_path_count += self.shortest_path_log_count(
                agent_index,
                next_position,
            )
            total_progress_options += self.progress_action_count(
                agent_index,
                next_position,
            )
            total_degree += self.local_degree(next_position)

        return (
            round(self_loop_probability, 12),
            round(conflict_probability, 12),
            deterministic_conflicts,
            foreign_goal_occupancy,
            nonprogress_count,
            stationary_count,
            round(-total_log_path_count, 12),
            -total_progress_options,
            -total_degree,
        )

    def od_operator_guidance_key(
        self,
        state: State,
        prefix: Sequence[Action],
        action: Action,
    ) -> GuidanceKey:
        """
        Secondary key for tied OD operators.

        A complete prefix receives the same joint-action guidance as Baseline.
        For an incomplete prefix, the key evaluates the next agent's progress,
        shortest-path diversity, mobility, and conflicts with already fixed
        actions or still-occupied cells.
        """
        self.mdp.validate_state(state)

        if action not in ACTIONS:
            raise ValueError(f"Unknown action: {action!r}")

        extended_prefix = tuple(prefix) + (action,)

        if len(extended_prefix) > self.mdp.n_agents:
            raise ValueError("OD prefix is longer than the number of agents")

        if len(extended_prefix) == self.mdp.n_agents:
            return self.joint_action_guidance_key(
                state,
                extended_prefix,
            )

        optimistic_self_loop_probability = (
            self.optimistic_prefix_self_loop_probability(
                state,
                extended_prefix,
            )
        )

        current_agent = len(prefix)
        current_position = state[current_agent]
        intended_position = self.deterministic_result(
            current_agent,
            current_position,
            action,
        )

        fixed_intended_positions = tuple(
            self.deterministic_result(
                agent_index,
                state[agent_index],
                fixed_action,
            )
            for agent_index, fixed_action in enumerate(extended_prefix)
        )

        partial_vertex_conflicts = self.mdp.vertex_conflict_count(
            fixed_intended_positions
        )
        partial_edge_swaps = self.mdp.edge_swap_conflict_count(
            state[: len(extended_prefix)],
            fixed_intended_positions,
        )

        unselected_current_positions = state[len(extended_prefix) :]
        occupancy_risk = sum(
            intended_position == other_position
            for other_position in unselected_current_positions
        )

        foreign_goal_occupancy = sum(
            intended_position == other_goal
            for other_index, other_goal in enumerate(self.mdp.goals)
            if other_index != current_agent
        )

        current_distance = self.agent_distance(
            current_agent,
            current_position,
        )
        next_distance = self.agent_distance(
            current_agent,
            intended_position,
        )

        nonprogress = int(
            current_position != self.mdp.goals[current_agent]
            and next_distance >= current_distance
        )
        stationary = int(
            current_position != self.mdp.goals[current_agent]
            and intended_position == current_position
        )

        return (
            round(optimistic_self_loop_probability, 12),
            partial_vertex_conflicts + partial_edge_swaps,
            occupancy_risk,
            foreign_goal_occupancy,
            nonprogress,
            stationary,
            round(
                -self.shortest_path_log_count(
                    current_agent,
                    intended_position,
                ),
                12,
            ),
            -self.progress_action_count(
                current_agent,
                intended_position,
            ),
            -self.local_degree(intended_position),
        )

    def value_for_od_state(
        self,
        od_state: tuple[State, JointAction],
    ) -> float:
        state, prefix = od_state
        return self.od_value(state, prefix)

    def distance_summary(
        self,
        state: State,
    ) -> tuple[float, ...]:
        """Return deterministic shortest-path distances for limits and logs."""
        self.mdp.validate_state(state)

        return tuple(
            self.agent_distance(
                agent_index,
                position,
            )
            for agent_index, position in enumerate(state)
        )



"""
Baseline RTDP for the stochastic cooperative grid MMDP.

The planner solves a cost-minimization Stochastic Shortest Path problem:

    V(s) = min_a sum_s' P(s' | s, a)
           [c(s, a, s') + V(s')]

There is no discount factor.

For a state that has not yet received a Bellman backup, its value is
initialized using the obstacle-aware shortest-path heuristic.

This is the baseline implementation: every Bellman backup enumerates all
complete joint actions. Therefore, with five actions per agent, the branching
factor is:

    5 ** number_of_agents
"""

from dataclasses import dataclass
import hashlib
import math
import random
import time
from typing import Callable, Iterator



StateHeuristic = Callable[[State], float]


class _DeadlineReached(RuntimeError):
    """Internal signal used to stop planning when the time limit is reached."""


class _MemoryLimitReached(RuntimeError):
    """Internal signal used to stop planning at the configured RSS delta."""


@dataclass(frozen=True)
class RTDPConfig:
    """Configuration shared by Baseline RTDP and OD-RTDP.

    Algorithmic limits are optional.  ``step_tail_probability`` replaces the
    former fixed 5x step multiplier with a map-derived stochastic tail bound.
    ``memory_limit_mb`` is additional process RSS above the start of planning;
    final memory-limited experiments should isolate each run in a subprocess.
    """

    max_trials: int | None = None
    max_steps_per_trial: int | None = None

    # Backward-compatible override. None uses the probabilistic bound.
    step_limit_multiplier: float | None = None
    step_tail_probability: float = 1e-6

    time_limit_seconds: float | None = 60.0
    memory_limit_mb: float | None = None

    # Scale-aware Bellman-residual criterion.
    epsilon: float = 1e-8
    relative_epsilon: float = 1e-6
    stable_trials_required: int = 44
    stop_when_stable: bool = False
    # LRTDP-style stopping: stop when the initial state is labelled solved.
    # This is the preferred rule for run-to-convergence experiments.
    stop_when_solved: bool = False
    require_goal_for_stability: bool = True

    # None means an ULP-based numerical comparison; a positive value keeps the
    # old explicit absolute-tolerance behavior for sensitivity experiments.
    tie_tolerance: float | None = None
    tie_ulps: int = 8
    seed: int = 0

    def __post_init__(self) -> None:
        if self.max_trials is not None and self.max_trials <= 0:
            raise ValueError("max_trials must be positive or None")
        if self.max_steps_per_trial is not None and self.max_steps_per_trial <= 0:
            raise ValueError("max_steps_per_trial must be positive or None")
        if self.step_limit_multiplier is not None and self.step_limit_multiplier <= 0.0:
            raise ValueError("step_limit_multiplier must be positive or None")
        if not 0.0 < self.step_tail_probability < 1.0:
            raise ValueError("step_tail_probability must be in (0, 1)")
        if self.time_limit_seconds is not None and self.time_limit_seconds <= 0.0:
            raise ValueError("time_limit_seconds must be positive or None")
        if self.memory_limit_mb is not None and self.memory_limit_mb <= 0.0:
            raise ValueError("memory_limit_mb must be positive or None")
        if self.epsilon < 0.0 or self.relative_epsilon < 0.0:
            raise ValueError("residual tolerances cannot be negative")
        if self.stable_trials_required <= 0:
            raise ValueError("stable_trials_required must be positive")
        if self.tie_tolerance is not None and self.tie_tolerance < 0.0:
            raise ValueError("tie_tolerance cannot be negative or None")
        if self.tie_ulps <= 0:
            raise ValueError("tie_ulps must be positive")
        if (
            self.max_trials is None
            and self.time_limit_seconds is None
            and self.memory_limit_mb is None
            and not self.stop_when_stable
            and not self.stop_when_solved
        ):
            raise ValueError(
                "At least one stopping mechanism must be enabled: max_trials, "
                "time_limit_seconds, memory_limit_mb, stop_when_stable, "
                "or stop_when_solved"
            )

@dataclass(frozen=True)
class TrialResult:
    maximum_residual: float
    maximum_scaled_residual: float
    steps: int
    reached_goal: bool
    reached_solved_state: bool
    hit_step_limit: bool
    visited_states: tuple[State, ...]


@dataclass(frozen=True)
class RTDPPlanningResult:
    stop_reason: str
    trials_completed: int
    goal_reaching_trials: int
    step_limited_trials: int
    total_trial_steps: int
    elapsed_seconds: float
    bellman_backups: int
    planning_action_evaluations: int
    transition_outcomes_evaluated: int
    visited_states: int
    final_trial_residual: float
    final_trial_scaled_residual: float
    consecutive_stable_trials: int
    maximum_consecutive_stable_trials: int
    stability_criterion_reached: bool
    first_stability_trial: int | None
    first_stability_elapsed_seconds: float | None
    initial_state_solved: bool
    solved_states: int
    solved_checks: int
    first_solved_trial: int | None
    first_solved_elapsed_seconds: float | None
    resolved_max_steps_per_trial: int | None
    memory_limit_mb: float | None
    memory_limit_reached: bool
    baseline_rss_mb: float
    peak_rss_mb: float
    peak_rss_delta_mb: float


class BaselineRTDP:
    """
    Standard RTDP over complete states and complete joint actions.

    At each visited state, the planner:

    1. Enumerates every complete joint action.
    2. Computes the expected cost of each action.
    3. Selects an action with minimum expected cost.
    4. Stores the resulting Bellman value in V.
    5. Samples one stochastic successor.
    6. Continues the internal trial from that successor.
    """

    def __init__(
        self,
        mdp: GridMMDP,
        heuristic: StateHeuristic | None = None,
        config: RTDPConfig | None = None,
    ) -> None:
        self.mdp = mdp

        self.heuristic = (
            heuristic
            if heuristic is not None
            else ShortestPathHeuristic(mdp)
        )

        self.config = config or RTDPConfig()

        # Calculate the actual per-trial step limit once for this instance.
        self.resolved_max_steps_per_trial = (
            self._resolve_max_steps_per_trial()
        )

        # Store only states that have received a Bellman backup.
        self.V: dict[State, float] = {}

        # LRTDP solved labels. Terminal states are treated as solved without
        # being inserted into this set.
        self._solved_states: set[State] = set()
        self.solved_checks = 0

        # During evaluation, cache the complete set of greedy joint actions
        # for each real state. The selected action is still sampled from this
        # set on every visit when evaluation uses stochastic tie breaking.
        self._policy_candidate_cache: dict[
            State,
            tuple[JointAction, ...],
        ] = {}
        self._policy_cache_hits = 0
        self._policy_cache_misses = 0

        self.bellman_backups = 0
        self.planning_action_evaluations = 0
        self.transition_outcomes_evaluated = 0

        # Separate generators make transition sampling independent of the
        # number of random tie-breaking decisions.
        self.transition_rng = random.Random(
            self.config.seed
        )
        self.tie_rng = random.Random(
            self.config.seed + 1
        )
        self._resource_monitor: ResourceMonitor | None = None

    def _resolve_max_steps_per_trial(self) -> int | None:
        """Resolve a finite, map-derived safety cap for one sampled trial."""
        if self.config.max_steps_per_trial is not None:
            return self.config.max_steps_per_trial

        distance_summary_method = getattr(self.heuristic, "distance_summary", None)
        if not callable(distance_summary_method):
            if self.config.step_limit_multiplier is None:
                return None
            raise ValueError(
                "Automatic step-limit calculation requires distance_summary"
            )

        distances = distance_summary_method(self.mdp.initial_state())
        if any(math.isinf(distance) for distance in distances):
            raise ValueError("At least one start cannot reach its assigned goal")
        success_probability = 1.0 - self.mdp.config.slip_to_stay_probability
        if success_probability <= 0.0:
            raise ValueError("Movement success probability must be positive")

        if self.config.step_limit_multiplier is not None:
            longest = max(distances, default=0.0)
            return max(
                1,
                math.ceil(
                    self.config.step_limit_multiplier
                    * longest
                    / success_probability
                ),
            )

        return sequential_multi_agent_step_bound(
            distances,
            success_probability,
            self.config.step_tail_probability,
        )

    def reset(self) -> None:
        """
        Remove the current solution and restore counters and RNG states.
        """
        self.V.clear()
        self._solved_states.clear()
        self.solved_checks = 0
        self._policy_candidate_cache.clear()
        self.reset_policy_cache_stats()

        self.bellman_backups = 0
        self.planning_action_evaluations = 0
        self.transition_outcomes_evaluated = 0

        self.transition_rng.seed(
            self.config.seed
        )
        self.tie_rng.seed(
            self.config.seed + 1
        )

    def _check_deadline(self, deadline: float | None) -> None:
        if deadline is not None and time.perf_counter() >= deadline:
            raise _DeadlineReached
        if (
            self._resource_monitor is not None
            and self._resource_monitor.limit_reached()
        ):
            raise _MemoryLimitReached

    def _values_tied(self, first: float, second: float) -> bool:
        if self.config.tie_tolerance is not None:
            return math.isclose(
                first, second, rel_tol=0.0, abs_tol=self.config.tie_tolerance
            )
        return tied_by_ulp(first, second, ulps=self.config.tie_ulps)

    def value(
        self,
        state: State,
    ) -> float:
        """
        Return the current estimate of V(state).

        Terminal state:
            0

        State already updated:
            Stored Bellman value

        State not yet updated:
            Shortest-path heuristic
        """
        if self.mdp.is_terminal(state):
            return 0.0

        if state in self.V:
            return self.V[state]

        return self.heuristic(state)

    def q_value(
        self,
        state: State,
        joint_action: JointAction,
        *,
        count_metrics: bool = True,
        deadline: float | None = None,
    ) -> float:
        """
        Compute the expected cost of one complete joint action.

            Q(s,a) =
                sum_s' P(s'|s,a)
                [transition_cost(s,a,s') + V(s')]

        If a successor has not been updated yet, V(s') is initialized by the
        shortest-path heuristic.
        """
        self._check_deadline(
            deadline
        )

        transitions = self.mdp.joint_transitions(
            state,
            joint_action,
        )

        self._check_deadline(
            deadline
        )

        if count_metrics:
            self.planning_action_evaluations += 1
            self.transition_outcomes_evaluated += len(
                transitions
            )

        expected_cost = 0.0

        for next_state, probability in transitions:
            self._check_deadline(
                deadline
            )

            immediate_cost = self.mdp.transition_cost(
                state,
                joint_action,
                next_state,
            )

            future_cost = self.value(
                next_state
            )

            expected_cost += probability * (
                immediate_cost + future_cost
            )

        return expected_cost

    def _deterministic_tie_choice(
        self,
        candidates: list[JointAction],
        state: State,
    ) -> JointAction:
        """
        Choose reproducibly among equal-valued joint actions.

        The choice depends on the planning seed, the current state, and the
        tied candidates. This avoids always preferring the first action in
        ACTIONS (usually ``stay``) while keeping evaluation deterministic.
        """
        if not candidates:
            raise ValueError(
                "candidates cannot be empty"
            )

        payload = repr(
            (
                self.config.seed,
                state,
                tuple(candidates),
            )
        ).encode("utf-8")

        digest = hashlib.sha256(
            payload
        ).digest()

        index = int.from_bytes(
            digest[:8],
            byteorder="big",
        ) % len(candidates)

        return candidates[index]

    def _guided_joint_actions(
        self,
        state: State,
        candidates: list[JointAction],
    ) -> list[JointAction]:
        """
        Refine a Q-tied set using structured heuristic guidance.

        Bellman/Q values remain the primary criterion. The heuristic key is
        consulted only after actions are equal within tie_tolerance, so this
        reduces arbitrary shortest-path ties without perturbing the objective.
        Custom heuristics that do not expose joint_action_guidance_key keep the
        original candidate set unchanged.
        """
        if len(candidates) <= 1:
            return candidates

        guidance_method = getattr(
            self.heuristic,
            "joint_action_guidance_key",
            None,
        )

        if not callable(guidance_method):
            return candidates

        keyed_candidates = [
            (guidance_method(state, action), action)
            for action in candidates
        ]
        best_key = min(
            key
            for key, _ in keyed_candidates
        )

        return [
            action
            for key, action in keyed_candidates
            if key == best_key
        ]

    def best_action(
        self,
        state: State,
        *,
        count_metrics: bool = True,
        random_ties: bool = True,
        deadline: float | None = None,
    ) -> tuple[JointAction, float]:
        """
        Return a minimum-expected-cost complete joint action.
        """
        if self.mdp.is_terminal(state):
            stay_action: JointAction = tuple(
                "stay"
                for _ in range(self.mdp.n_agents)
            )
            return stay_action, 0.0

        best_value = math.inf
        best_actions: list[JointAction] = []

        for joint_action in self.mdp.all_joint_actions():
            self._check_deadline(
                deadline
            )

            action_value = self.q_value(
                state,
                joint_action,
                count_metrics=count_metrics,
                deadline=deadline,
            )

            if action_value < best_value and not self._values_tied(
                action_value, best_value
            ):
                best_value = action_value
                best_actions = [
                    joint_action
                ]

            elif self._values_tied(action_value, best_value):
                best_actions.append(
                    joint_action
                )

        if not best_actions:
            raise RuntimeError(
                "No joint action was generated"
            )

        best_actions = self._guided_joint_actions(
            state,
            best_actions,
        )

        if random_ties:
            selected_action = self.tie_rng.choice(
                best_actions
            )
        else:
            selected_action = self._deterministic_tie_choice(
                best_actions,
                state,
            )

        return selected_action, best_value

    def backup(
        self,
        state: State,
        *,
        deadline: float | None = None,
    ) -> tuple[JointAction, float, float]:
        old_value = self.value(state)
        joint_action, new_value = self.best_action(
            state, count_metrics=True, random_ties=True, deadline=deadline
        )
        self.V[state] = new_value
        self.bellman_backups += 1
        residual = abs(new_value - old_value)
        scaled = scaled_residual_ratio(
            old_value,
            new_value,
            absolute_tolerance=self.config.epsilon,
            relative_tolerance=self.config.relative_epsilon,
        )
        return joint_action, residual, scaled

    def _trial_step_numbers(
        self,
    ) -> Iterator[int]:
        """
        Generate step indices for one internal trial.

        If resolved_max_steps_per_trial is None, the iterator is unbounded.
        """
        step_number = 0

        while (
            self.resolved_max_steps_per_trial is None
            or step_number
            < self.resolved_max_steps_per_trial
        ):
            yield step_number
            step_number += 1

    def run_trial(
        self,
        *,
        deadline: float | None = None,
    ) -> TrialResult:
        state = self.mdp.initial_state()
        maximum_residual = 0.0
        maximum_scaled_residual = 0.0
        steps = 0
        visited_states: list[State] = []

        if self.mdp.is_terminal(state):
            return TrialResult(0.0, 0.0, 0, True, False, False, ())

        for _ in self._trial_step_numbers():
            self._check_deadline(deadline)

            # In LRTDP mode there is no reason to continue through a state
            # whose greedy envelope has already been proved solved.
            if self.config.stop_when_solved and state in self._solved_states:
                return TrialResult(
                    maximum_residual,
                    maximum_scaled_residual,
                    steps,
                    False,
                    True,
                    False,
                    tuple(visited_states),
                )

            visited_states.append(state)
            joint_action, residual, scaled = self.backup(
                state, deadline=deadline
            )
            maximum_residual = max(maximum_residual, residual)
            maximum_scaled_residual = max(maximum_scaled_residual, scaled)
            state = self.mdp.sample_next(
                state, joint_action, self.transition_rng
            )
            steps += 1
            if self.mdp.is_terminal(state):
                return TrialResult(
                    maximum_residual,
                    maximum_scaled_residual,
                    steps,
                    True,
                    False,
                    False,
                    tuple(visited_states),
                )

        return TrialResult(
            maximum_residual,
            maximum_scaled_residual,
            steps,
            False,
            False,
            True,
            tuple(visited_states),
        )

    def _greedy_successors_for_solved_check(
        self,
        state: State,
        action: JointAction,
    ) -> tuple[State, ...]:
        """Return all positive-probability successors of one fixed greedy action."""
        transitions = self.mdp.joint_transitions(state, action)
        return tuple(
            next_state
            for next_state, probability in transitions
            if probability > 0.0
        )

    def check_solved(
        self,
        root: State,
        *,
        deadline: float | None = None,
    ) -> bool:
        """LRTDP-style solved-state check for the greedy policy envelope.

        A nonterminal state is labelled solved only when its Bellman residual
        is within the configured scale-aware tolerance and every
        positive-probability successor of the deterministic greedy action is
        terminal, already solved, or belongs to the same locally consistent
        envelope.  If the envelope is not solved, reverse Bellman backups are
        performed to propagate information before the next trial.
        """
        self.solved_checks += 1
        if self.mdp.is_terminal(root) or root in self._solved_states:
            return True

        open_stack: list[State] = [root]
        open_set: set[State] = {root}
        closed: list[State] = []
        closed_set: set[State] = set()
        envelope_is_solved = True

        while open_stack:
            self._check_deadline(deadline)
            state = open_stack.pop()
            open_set.discard(state)
            if self.mdp.is_terminal(state) or state in self._solved_states:
                continue
            if state in closed_set:
                continue

            closed.append(state)
            closed_set.add(state)

            old_value = self.value(state)
            action, bellman_value = self.best_action(
                state,
                count_metrics=True,
                random_ties=False,
                deadline=deadline,
            )
            scaled = scaled_residual_ratio(
                old_value,
                bellman_value,
                absolute_tolerance=self.config.epsilon,
                relative_tolerance=self.config.relative_epsilon,
            )
            if scaled > 1.0:
                envelope_is_solved = False
                continue

            for successor in self._greedy_successors_for_solved_check(
                state, action
            ):
                if (
                    self.mdp.is_terminal(successor)
                    or successor in self._solved_states
                    or successor in closed_set
                    or successor in open_set
                ):
                    continue
                open_stack.append(successor)
                open_set.add(successor)

        if envelope_is_solved:
            self._solved_states.update(closed_set)
            return True

        # Standard LRTDP repair step: back up the inconsistent envelope in
        # reverse discovery order, then continue sampling on the next trial.
        for state in reversed(closed):
            self._check_deadline(deadline)
            if state not in self._solved_states:
                self.backup(state, deadline=deadline)
        return False

    def _label_trial_path(
        self,
        visited_states: tuple[State, ...],
        *,
        deadline: float | None = None,
    ) -> None:
        """Try to label states on a sampled trial path, from end to start."""
        for state in reversed(visited_states):
            self._check_deadline(deadline)
            if not self.check_solved(state, deadline=deadline):
                break

    def _trial_numbers(
        self,
    ) -> Iterator[int]:
        """
        Generate bounded or unbounded internal trial numbers.
        """
        trial_number = 1

        while (
            self.config.max_trials is None
            or trial_number <= self.config.max_trials
        ):
            yield trial_number
            trial_number += 1

    def solve(self, *, reset: bool = True) -> RTDPPlanningResult:
        if reset:
            self.reset()
        self._policy_candidate_cache.clear()
        self.reset_policy_cache_stats()

        started_at = time.perf_counter()
        deadline = (
            started_at + self.config.time_limit_seconds
            if self.config.time_limit_seconds is not None
            else None
        )
        monitor = ResourceMonitor(memory_limit_mb=self.config.memory_limit_mb)
        self._resource_monitor = monitor.start()

        trials_completed = 0
        goal_reaching_trials = 0
        step_limited_trials = 0
        total_trial_steps = 0
        consecutive_stable_trials = 0
        maximum_consecutive_stable_trials = 0
        first_stability_trial: int | None = None
        first_stability_elapsed_seconds: float | None = None
        first_solved_trial: int | None = None
        first_solved_elapsed_seconds: float | None = None
        final_trial_residual = math.inf
        final_trial_scaled_residual = math.inf
        stop_reason = "max_trials"

        try:
            for trial_number in self._trial_numbers():
                try:
                    trial_result = self.run_trial(deadline=deadline)
                except _DeadlineReached:
                    stop_reason = "time_limit"
                    break
                except _MemoryLimitReached:
                    stop_reason = "memory_limit"
                    break

                trials_completed += 1
                total_trial_steps += trial_result.steps
                final_trial_residual = trial_result.maximum_residual
                final_trial_scaled_residual = (
                    trial_result.maximum_scaled_residual
                )
                if trial_result.reached_goal:
                    goal_reaching_trials += 1
                if trial_result.hit_step_limit:
                    step_limited_trials += 1

                stable = (
                    trial_result.maximum_scaled_residual <= 1.0
                    and (
                        trial_result.reached_goal
                        or not self.config.require_goal_for_stability
                    )
                )
                consecutive_stable_trials = (
                    consecutive_stable_trials + 1 if stable else 0
                )
                maximum_consecutive_stable_trials = max(
                    maximum_consecutive_stable_trials,
                    consecutive_stable_trials,
                )
                if (
                    first_stability_trial is None
                    and consecutive_stable_trials
                    >= self.config.stable_trials_required
                ):
                    first_stability_trial = trial_number
                    first_stability_elapsed_seconds = (
                        time.perf_counter() - started_at
                    )
                if self.config.stop_when_solved:
                    try:
                        self._label_trial_path(
                            trial_result.visited_states,
                            deadline=deadline,
                        )
                    except _DeadlineReached:
                        stop_reason = "time_limit"
                        break
                    except _MemoryLimitReached:
                        stop_reason = "memory_limit"
                        break

                    initial_state = self.mdp.initial_state()
                    if (
                        self.mdp.is_terminal(initial_state)
                        or initial_state in self._solved_states
                    ):
                        if first_solved_trial is None:
                            first_solved_trial = trial_number
                            first_solved_elapsed_seconds = (
                                time.perf_counter() - started_at
                            )
                        stop_reason = "initial_state_solved"
                        break

                if (
                    self.config.stop_when_stable
                    and consecutive_stable_trials
                    >= self.config.stable_trials_required
                ):
                    stop_reason = "stable_trials"
                    break
                self._check_deadline(deadline)
        finally:
            snapshot = monitor.stop()
            self._resource_monitor = None

        elapsed_seconds = time.perf_counter() - started_at
        return RTDPPlanningResult(
            stop_reason=stop_reason,
            trials_completed=trials_completed,
            goal_reaching_trials=goal_reaching_trials,
            step_limited_trials=step_limited_trials,
            total_trial_steps=total_trial_steps,
            elapsed_seconds=elapsed_seconds,
            bellman_backups=self.bellman_backups,
            planning_action_evaluations=self.planning_action_evaluations,
            transition_outcomes_evaluated=self.transition_outcomes_evaluated,
            visited_states=len(self.V),
            final_trial_residual=final_trial_residual,
            final_trial_scaled_residual=final_trial_scaled_residual,
            consecutive_stable_trials=consecutive_stable_trials,
            maximum_consecutive_stable_trials=(
                maximum_consecutive_stable_trials
            ),
            stability_criterion_reached=first_stability_trial is not None,
            first_stability_trial=first_stability_trial,
            first_stability_elapsed_seconds=first_stability_elapsed_seconds,
            initial_state_solved=(
                self.mdp.is_terminal(self.mdp.initial_state())
                or self.mdp.initial_state() in self._solved_states
            ),
            solved_states=len(self._solved_states),
            solved_checks=self.solved_checks,
            first_solved_trial=first_solved_trial,
            first_solved_elapsed_seconds=first_solved_elapsed_seconds,
            resolved_max_steps_per_trial=self.resolved_max_steps_per_trial,
            memory_limit_mb=self.config.memory_limit_mb,
            memory_limit_reached=snapshot.memory_limit_reached,
            baseline_rss_mb=snapshot.baseline_rss_mb,
            peak_rss_mb=snapshot.peak_rss_mb,
            peak_rss_delta_mb=snapshot.peak_rss_delta_mb,
        )

    def reset_policy_cache_stats(self) -> None:
        """Reset policy-candidate cache hit/miss counters."""
        self._policy_cache_hits = 0
        self._policy_cache_misses = 0

    def policy_cache_stats(self) -> dict[str, int | float]:
        """Return candidate-cache performance statistics."""
        total = self._policy_cache_hits + self._policy_cache_misses
        return {
            "hits": self._policy_cache_hits,
            "misses": self._policy_cache_misses,
            "entries": len(self._policy_candidate_cache),
            "hit_rate": (self._policy_cache_hits / total if total else 0.0),
        }

    def greedy_action_candidates(
        self,
        state: State,
    ) -> tuple[JointAction, ...]:
        """
        Return every joint action tied for the minimum current Q-value.

        The candidate set is memoized after planning because V is fixed during
        evaluation. Unlike the previous single-action cache, this preserves
        all greedy choices and allows reproducible stochastic tie breaking on
        repeated visits to the same state.
        """
        self.mdp.validate_state(state)

        cached = self._policy_candidate_cache.get(state)
        if cached is not None:
            self._policy_cache_hits += 1
            return cached

        self._policy_cache_misses += 1

        if self.mdp.is_terminal(state):
            terminal_candidates = (
                tuple("stay" for _ in range(self.mdp.n_agents)),
            )
            self._policy_candidate_cache[state] = terminal_candidates
            return terminal_candidates

        best_value = math.inf
        best_actions: list[JointAction] = []

        for joint_action in self.mdp.all_joint_actions():
            action_value = self.q_value(
                state,
                joint_action,
                count_metrics=False,
                deadline=None,
            )

            if action_value < best_value and not self._values_tied(action_value, best_value):
                best_value = action_value
                best_actions = [joint_action]
            elif self._values_tied(action_value, best_value):
                best_actions.append(joint_action)

        if not best_actions:
            raise RuntimeError("No joint action was generated")

        best_actions = self._guided_joint_actions(
            state,
            best_actions,
        )

        candidates = tuple(best_actions)
        self._policy_candidate_cache[state] = candidates
        return candidates

    def policy_action_with_info(
        self,
        state: State,
        *,
        tie_rng: random.Random | None = None,
    ) -> tuple[JointAction, int]:
        """
        Select a greedy action and report whether a tie had to be broken.

        When ``tie_rng`` is supplied, equal-valued actions are sampled on every
        visit. Without it, a stable hash provides deterministic backward-
        compatible behavior.
        """
        candidates = self.greedy_action_candidates(state)
        tie_decisions = int(len(candidates) > 1)

        if len(candidates) == 1:
            return candidates[0], tie_decisions

        if tie_rng is not None:
            return tie_rng.choice(candidates), tie_decisions

        return (
            self._deterministic_tie_choice(list(candidates), state),
            tie_decisions,
        )

    def policy_action(
        self,
        state: State,
        *,
        tie_rng: random.Random | None = None,
    ) -> JointAction:
        """
        Return a current greedy policy action without updating V or counters.

        Supplying ``tie_rng`` creates a stochastic greedy policy over actions
        that are equal within ``tie_tolerance``. This prevents a fixed cached
        tie choice from trapping evaluation in an arbitrary cycle.
        """
        action, _ = self.policy_action_with_info(
            state,
            tie_rng=tie_rng,
        )
        return action



"""
Operator-Decomposition RTDP for the stochastic cooperative grid MMDP.

The planner solves the same cost-minimization Stochastic Shortest Path problem
as Baseline RTDP:

    V(s) = min_a sum_s' P(s' | s, a)
           [c(s, a, s') + V(s')]

The difference is how a complete joint action is selected.

Baseline RTDP evaluates every complete joint action at once. With five actions
per agent, one Bellman backup has branching factor 5 ** number_of_agents.

OD-RTDP represents an intermediate planning state as:

    (real_state, action_prefix)

and chooses one agent action at a time. Every intermediate OD state has only
five outgoing operators. No real time, cost, stochastic transition, or slip is
applied while the prefix is being built. Only after the prefix contains one
action for every agent is the complete joint action executed in the real MMDP.
"""

from dataclasses import dataclass
import hashlib
import math
import random
import time
from typing import Iterator



# A stored OD state always has an incomplete action prefix. A complete prefix is
# executed immediately in the real environment and is not stored as an OD state.
ODState = tuple[State, JointAction]


class _DeadlineReached(RuntimeError):
    """Internal signal used to stop planning when the deadline is reached."""


class _MemoryLimitReached(RuntimeError):
    """Internal signal used to stop planning at the RSS-delta limit."""


@dataclass(frozen=True)
class ODTrialResult:
    maximum_residual: float
    maximum_scaled_residual: float
    real_steps: int
    reached_goal: bool
    reached_solved_state: bool
    hit_step_limit: bool
    visited_od_states: tuple[ODState, ...]


@dataclass(frozen=True)
class ODRTDPPlanningResult:
    stop_reason: str
    trials_completed: int
    goal_reaching_trials: int
    step_limited_trials: int
    total_real_steps: int
    elapsed_seconds: float
    bellman_backups: int
    planning_operator_evaluations: int
    complete_joint_actions_evaluated: int
    transition_outcomes_evaluated: int
    visited_od_states: int
    visited_real_states: int
    final_trial_residual: float
    final_trial_scaled_residual: float
    consecutive_stable_trials: int
    maximum_consecutive_stable_trials: int
    stability_criterion_reached: bool
    first_stability_trial: int | None
    first_stability_elapsed_seconds: float | None
    initial_state_solved: bool
    solved_od_states: int
    solved_real_states: int
    solved_checks: int
    first_solved_trial: int | None
    first_solved_elapsed_seconds: float | None
    resolved_max_steps_per_trial: int | None
    memory_limit_mb: float | None
    memory_limit_reached: bool
    baseline_rss_mb: float
    peak_rss_mb: float
    peak_rss_delta_mb: float


class OperatorDecompositionRTDP:
    """
    RTDP over operator-decomposition states.

    For an incomplete prefix alpha, the Bellman equation is:

        V_OD(s, alpha) = min_a_i V_OD(s, alpha + a_i)

    because choosing another component of the joint action does not yet consume
    a real environment step.

    When alpha + a_i is a complete joint action, the real MMDP equation is used:

        Q_OD(s, alpha, a_i)
            = sum_s' P(s' | s, alpha + a_i)
              [c(s, alpha + a_i, s') + V_OD(s', empty_prefix)]

    Thus Baseline RTDP and OD-RTDP optimize exactly the same objective and use
    exactly the same environment model. Only the action-selection structure is
    different.
    """

    def __init__(
        self,
        mdp: GridMMDP,
        heuristic: ShortestPathHeuristic | None = None,
        config: RTDPConfig | None = None,
    ) -> None:
        self.mdp = mdp
        self.heuristic = (
            heuristic
            if heuristic is not None
            else ShortestPathHeuristic(mdp)
        )
        self.config = config or RTDPConfig()

        self.resolved_max_steps_per_trial = (
            self._resolve_max_steps_per_trial()
        )

        # Values are stored for real states paired with incomplete prefixes.
        self.V: dict[ODState, float] = {}

        # LRTDP solved labels on the expanded OD state space. Terminal real
        # states with an empty prefix are treated as solved implicitly.
        self._solved_od_states: set[ODState] = set()
        self.solved_checks = 0

        # Evaluation uses three immutable caches after planning:
        #
        # 1. Bellman-tied local operators before secondary guidance.
        # 2. Locally guided operators for the public diagnostic method.
        # 3. Globally guided complete joint actions for the actual policy.
        #
        # The third cache avoids rebuilding a complete action one prefix at a
        # time on every visit.  More importantly, complete-action guidance is
        # applied only after all Bellman-tied prefixes are expanded, so an
        # early local tie cannot accidentally force a high-self-loop joint
        # action.
        self._raw_operator_candidate_cache: dict[
            ODState,
            tuple[Action, ...],
        ] = {}
        self._guided_operator_candidate_cache: dict[
            ODState,
            tuple[Action, ...],
        ] = {}
        self._joint_policy_candidate_cache: dict[
            State,
            tuple[JointAction, ...],
        ] = {}
        self._global_real_policy_candidate_cache: dict[
            State,
            tuple[JointAction, ...],
        ] = {}
        self._policy_cache_hits = 0
        self._policy_cache_misses = 0

        self.bellman_backups = 0
        self.planning_operator_evaluations = 0
        self.complete_joint_actions_evaluated = 0
        self.transition_outcomes_evaluated = 0

        # Transition sampling is deliberately independent from tie breaking.
        self.transition_rng = random.Random(self.config.seed)
        self.tie_rng = random.Random(self.config.seed + 1)
        self._resource_monitor: ResourceMonitor | None = None

    def _resolve_max_steps_per_trial(self) -> int | None:
        if self.config.max_steps_per_trial is not None:
            return self.config.max_steps_per_trial
        distances = self.heuristic.distance_summary(self.mdp.initial_state())
        if any(math.isinf(distance) for distance in distances):
            raise ValueError("At least one start cannot reach its assigned goal")
        success_probability = 1.0 - self.mdp.config.slip_to_stay_probability
        if success_probability <= 0.0:
            raise ValueError("Movement success probability must be positive")
        if self.config.step_limit_multiplier is not None:
            longest = max(distances, default=0.0)
            return max(
                1,
                math.ceil(
                    self.config.step_limit_multiplier
                    * longest
                    / success_probability
                ),
            )
        return sequential_multi_agent_step_bound(
            distances,
            success_probability,
            self.config.step_tail_probability,
        )

    def reset(self) -> None:
        """Remove the current solution and reset counters and random states."""
        self.V.clear()
        self._solved_od_states.clear()
        self.solved_checks = 0
        self._raw_operator_candidate_cache.clear()
        self._guided_operator_candidate_cache.clear()
        self._joint_policy_candidate_cache.clear()
        self._global_real_policy_candidate_cache.clear()
        self.reset_policy_cache_stats()

        self.bellman_backups = 0
        self.planning_operator_evaluations = 0
        self.complete_joint_actions_evaluated = 0
        self.transition_outcomes_evaluated = 0

        self.transition_rng.seed(self.config.seed)
        self.tie_rng.seed(self.config.seed + 1)

    def _check_deadline(self, deadline: float | None) -> None:
        if deadline is not None and time.perf_counter() >= deadline:
            raise _DeadlineReached
        if (
            self._resource_monitor is not None
            and self._resource_monitor.limit_reached()
        ):
            raise _MemoryLimitReached

    def _values_tied(self, first: float, second: float) -> bool:
        if self.config.tie_tolerance is not None:
            return math.isclose(
                first, second, rel_tol=0.0, abs_tol=self.config.tie_tolerance
            )
        return tied_by_ulp(first, second, ulps=self.config.tie_ulps)

    def validate_od_state(self, od_state: ODState) -> None:
        """Validate a real state paired with an incomplete action prefix."""
        state, prefix = od_state
        self.mdp.validate_state(state)

        if len(prefix) >= self.mdp.n_agents:
            raise ValueError(
                "A stored OD prefix must be incomplete; complete joint "
                "actions are executed immediately."
            )

        invalid_actions = [
            action for action in prefix if action not in ACTIONS
        ]
        if invalid_actions:
            raise ValueError(
                f"OD prefix contains unknown actions: {invalid_actions}"
            )

        if self.mdp.is_terminal(state) and prefix:
            raise ValueError(
                "A terminal real state must use an empty OD prefix."
            )

    def value(self, od_state: ODState) -> float:
        """
        Return the current value estimate of an OD state.

        Terminal real state:
            0

        Previously updated OD state:
            Stored Bellman value

        New OD state:
            Operator-decomposition shortest-path heuristic
        """
        self.validate_od_state(od_state)
        state, prefix = od_state

        if self.mdp.is_terminal(state):
            return 0.0

        if od_state in self.V:
            return self.V[od_state]

        return self.heuristic.od_value(state, prefix)

    def real_state_value(self, state: State) -> float:
        """Return the value of a real state with an empty action prefix."""
        return self.value((state, ()))

    def complete_joint_action_value(
        self,
        state: State,
        joint_action: JointAction,
        *,
        count_metrics: bool = True,
        deadline: float | None = None,
    ) -> float:
        """
        Evaluate a complete joint action using the real stochastic MMDP.

        Q(s,a) = sum_s' P(s'|s,a) [c(s,a,s') + V_OD(s', empty)]
        """
        self._check_deadline(deadline)
        self.mdp.validate_state(state)
        self.mdp.validate_joint_action(joint_action)

        transitions = self.mdp.joint_transitions(
            state,
            joint_action,
        )

        self._check_deadline(deadline)

        if count_metrics:
            self.complete_joint_actions_evaluated += 1
            self.transition_outcomes_evaluated += len(transitions)

        expected_cost = 0.0

        for next_state, probability in transitions:
            self._check_deadline(deadline)

            immediate_cost = self.mdp.transition_cost(
                state,
                joint_action,
                next_state,
            )
            future_cost = self.real_state_value(next_state)

            expected_cost += probability * (
                immediate_cost + future_cost
            )

        return expected_cost

    def operator_value(
        self,
        od_state: ODState,
        action: Action,
        *,
        count_metrics: bool = True,
        deadline: float | None = None,
    ) -> float:
        """
        Evaluate one local OD operator for the next agent.

        For an incomplete resulting prefix, no real transition occurs and the
        value is simply the value of the child OD state.

        For a complete resulting prefix, execute the complete joint action in
        the real stochastic MMDP and include the real transition cost.
        """
        self._check_deadline(deadline)
        self.validate_od_state(od_state)

        if action not in ACTIONS:
            raise ValueError(f"Unknown action: {action!r}")

        if count_metrics:
            self.planning_operator_evaluations += 1

        state, prefix = od_state
        extended_prefix = prefix + (action,)

        if len(extended_prefix) < self.mdp.n_agents:
            # Pure planning transition: no time, cost, slip, or collision.
            return self.value((state, extended_prefix))

        # The prefix is now a complete joint action and is executed for real.
        return self.complete_joint_action_value(
            state,
            extended_prefix,
            count_metrics=count_metrics,
            deadline=deadline,
        )

    def _deterministic_tie_choice(
        self,
        candidates: list[Action],
        od_state: ODState,
    ) -> Action:
        """
        Choose reproducibly among equal-valued OD operators.

        The real state, current prefix, planning seed, and tied actions define
        the choice. This prevents a fixed preference for the first action in
        ACTIONS while keeping policy evaluation deterministic.
        """
        if not candidates:
            raise ValueError(
                "candidates cannot be empty"
            )

        payload = repr(
            (
                self.config.seed,
                od_state,
                tuple(candidates),
            )
        ).encode("utf-8")

        digest = hashlib.sha256(
            payload
        ).digest()

        index = int.from_bytes(
            digest[:8],
            byteorder="big",
        ) % len(candidates)

        return candidates[index]

    def _guided_operators(
        self,
        od_state: ODState,
        candidates: list[Action],
    ) -> list[Action]:
        """
        Refine Bellman-tied OD operators using structured heuristic guidance.

        The operator value remains the primary criterion. Guidance is applied
        only within the tie_tolerance set, preserving the OD objective while
        distinguishing equally short branches by conflict risk, path diversity,
        future progress options, and local mobility.
        """
        if len(candidates) <= 1:
            return candidates

        guidance_method = getattr(
            self.heuristic,
            "od_operator_guidance_key",
            None,
        )

        if not callable(guidance_method):
            return candidates

        state, prefix = od_state
        keyed_candidates = [
            (guidance_method(state, prefix, action), action)
            for action in candidates
        ]
        best_key = min(
            key
            for key, _ in keyed_candidates
        )

        return [
            action
            for key, action in keyed_candidates
            if key == best_key
        ]

    def best_operator(
        self,
        od_state: ODState,
        *,
        count_metrics: bool = True,
        random_ties: bool = True,
        deadline: float | None = None,
    ) -> tuple[Action, float]:
        """Return the minimum-cost action for the next agent in the prefix."""
        self.validate_od_state(od_state)
        state, _ = od_state

        if self.mdp.is_terminal(state):
            return "stay", 0.0

        best_value = math.inf
        best_actions: list[Action] = []

        for action in ACTIONS:
            self._check_deadline(deadline)

            candidate_value = self.operator_value(
                od_state,
                action,
                count_metrics=count_metrics,
                deadline=deadline,
            )

            if candidate_value < best_value and not self._values_tied(
                candidate_value, best_value
            ):
                best_value = candidate_value
                best_actions = [action]

            elif self._values_tied(candidate_value, best_value):
                best_actions.append(action)

        if not best_actions:
            raise RuntimeError("No OD operator was generated")

        best_actions = self._guided_operators(
            od_state,
            best_actions,
        )

        if random_ties:
            selected_action = self.tie_rng.choice(
                best_actions
            )
        else:
            selected_action = self._deterministic_tie_choice(
                best_actions,
                od_state,
            )

        return selected_action, best_value

    def backup(
        self,
        od_state: ODState,
        *,
        deadline: float | None = None,
    ) -> tuple[Action, float, float]:
        old_value = self.value(od_state)
        selected_action, new_value = self.best_operator(
            od_state, count_metrics=True, random_ties=True, deadline=deadline
        )
        self.V[od_state] = new_value
        self.bellman_backups += 1
        residual = abs(new_value - old_value)
        scaled = scaled_residual_ratio(
            old_value,
            new_value,
            absolute_tolerance=self.config.epsilon,
            relative_tolerance=self.config.relative_epsilon,
        )
        return selected_action, residual, scaled

    def _trial_step_numbers(self) -> Iterator[int]:
        """Generate bounded or unbounded real environment step indices."""
        step_number = 0

        while (
            self.resolved_max_steps_per_trial is None
            or step_number < self.resolved_max_steps_per_trial
        ):
            yield step_number
            step_number += 1

    def run_trial(
        self,
        *,
        deadline: float | None = None,
    ) -> ODTrialResult:
        state = self.mdp.initial_state()
        maximum_residual = 0.0
        maximum_scaled_residual = 0.0
        real_steps = 0
        visited_od_states: list[ODState] = []
        if self.mdp.is_terminal(state):
            return ODTrialResult(0.0, 0.0, 0, True, False, False, ())

        for _ in self._trial_step_numbers():
            self._check_deadline(deadline)
            prefix: JointAction = ()
            while len(prefix) < self.mdp.n_agents:
                od_state = (state, prefix)
                if (
                    self.config.stop_when_solved
                    and od_state in self._solved_od_states
                ):
                    return ODTrialResult(
                        maximum_residual,
                        maximum_scaled_residual,
                        real_steps,
                        False,
                        True,
                        False,
                        tuple(visited_od_states),
                    )

                visited_od_states.append(od_state)
                selected_action, residual, scaled = self.backup(
                    od_state, deadline=deadline
                )
                maximum_residual = max(maximum_residual, residual)
                maximum_scaled_residual = max(
                    maximum_scaled_residual, scaled
                )
                prefix = prefix + (selected_action,)

            state = self.mdp.sample_next(state, prefix, self.transition_rng)
            real_steps += 1
            if self.mdp.is_terminal(state):
                return ODTrialResult(
                    maximum_residual,
                    maximum_scaled_residual,
                    real_steps,
                    True,
                    False,
                    False,
                    tuple(visited_od_states),
                )
        return ODTrialResult(
            maximum_residual,
            maximum_scaled_residual,
            real_steps,
            False,
            False,
            True,
            tuple(visited_od_states),
        )

    def _operator_successors_for_solved_check(
        self,
        od_state: ODState,
        action: Action,
    ) -> tuple[ODState, ...]:
        """Return every positive-probability OD successor of one operator."""
        state, prefix = od_state
        extended_prefix = prefix + (action,)
        if len(extended_prefix) < self.mdp.n_agents:
            return ((state, extended_prefix),)
        return tuple(
            (next_state, ())
            for next_state, probability in self.mdp.joint_transitions(
                state, extended_prefix
            )
            if probability > 0.0
        )

    def check_solved(
        self,
        root: ODState,
        *,
        deadline: float | None = None,
    ) -> bool:
        """LRTDP-style solved-state check on the expanded OD state space."""
        self.solved_checks += 1
        root_state, root_prefix = root
        if self.mdp.is_terminal(root_state):
            if root_prefix:
                raise ValueError("A terminal OD state must have an empty prefix")
            return True
        if root in self._solved_od_states:
            return True

        open_stack: list[ODState] = [root]
        open_set: set[ODState] = {root}
        closed: list[ODState] = []
        closed_set: set[ODState] = set()
        envelope_is_solved = True

        while open_stack:
            self._check_deadline(deadline)
            od_state = open_stack.pop()
            open_set.discard(od_state)
            state, prefix = od_state
            if self.mdp.is_terminal(state):
                if prefix:
                    raise ValueError(
                        "A terminal OD state must have an empty prefix"
                    )
                continue
            if od_state in self._solved_od_states or od_state in closed_set:
                continue

            closed.append(od_state)
            closed_set.add(od_state)

            old_value = self.value(od_state)
            action, bellman_value = self.best_operator(
                od_state,
                count_metrics=True,
                random_ties=False,
                deadline=deadline,
            )
            scaled = scaled_residual_ratio(
                old_value,
                bellman_value,
                absolute_tolerance=self.config.epsilon,
                relative_tolerance=self.config.relative_epsilon,
            )
            if scaled > 1.0:
                envelope_is_solved = False
                continue

            for successor in self._operator_successors_for_solved_check(
                od_state, action
            ):
                successor_state, successor_prefix = successor
                if self.mdp.is_terminal(successor_state):
                    if successor_prefix:
                        raise ValueError(
                            "A terminal OD state must have an empty prefix"
                        )
                    continue
                if (
                    successor in self._solved_od_states
                    or successor in closed_set
                    or successor in open_set
                ):
                    continue
                open_stack.append(successor)
                open_set.add(successor)

        if envelope_is_solved:
            self._solved_od_states.update(closed_set)
            return True

        for od_state in reversed(closed):
            self._check_deadline(deadline)
            if od_state not in self._solved_od_states:
                self.backup(od_state, deadline=deadline)
        return False

    def _label_trial_path(
        self,
        visited_od_states: tuple[ODState, ...],
        *,
        deadline: float | None = None,
    ) -> None:
        """Try to label OD states on a sampled path, from end to start."""
        for od_state in reversed(visited_od_states):
            self._check_deadline(deadline)
            if not self.check_solved(od_state, deadline=deadline):
                break

    def _trial_numbers(self) -> Iterator[int]:
        """Generate bounded or unbounded internal trial numbers."""
        trial_number = 1

        while (
            self.config.max_trials is None
            or trial_number <= self.config.max_trials
        ):
            yield trial_number
            trial_number += 1

    def solve(self, *, reset: bool = True) -> ODRTDPPlanningResult:
        if reset:
            self.reset()
        self._raw_operator_candidate_cache.clear()
        self._guided_operator_candidate_cache.clear()
        self._joint_policy_candidate_cache.clear()
        self._global_real_policy_candidate_cache.clear()
        self.reset_policy_cache_stats()

        started_at = time.perf_counter()
        deadline = (
            started_at + self.config.time_limit_seconds
            if self.config.time_limit_seconds is not None
            else None
        )
        monitor = ResourceMonitor(memory_limit_mb=self.config.memory_limit_mb)
        self._resource_monitor = monitor.start()

        trials_completed = 0
        goal_reaching_trials = 0
        step_limited_trials = 0
        total_real_steps = 0
        consecutive_stable_trials = 0
        maximum_consecutive_stable_trials = 0
        first_stability_trial: int | None = None
        first_stability_elapsed_seconds: float | None = None
        first_solved_trial: int | None = None
        first_solved_elapsed_seconds: float | None = None
        final_trial_residual = math.inf
        final_trial_scaled_residual = math.inf
        stop_reason = "max_trials"

        try:
            for trial_number in self._trial_numbers():
                try:
                    trial_result = self.run_trial(deadline=deadline)
                except _DeadlineReached:
                    stop_reason = "time_limit"
                    break
                except _MemoryLimitReached:
                    stop_reason = "memory_limit"
                    break

                trials_completed += 1
                total_real_steps += trial_result.real_steps
                final_trial_residual = trial_result.maximum_residual
                final_trial_scaled_residual = (
                    trial_result.maximum_scaled_residual
                )
                if trial_result.reached_goal:
                    goal_reaching_trials += 1
                if trial_result.hit_step_limit:
                    step_limited_trials += 1
                stable = (
                    trial_result.maximum_scaled_residual <= 1.0
                    and (
                        trial_result.reached_goal
                        or not self.config.require_goal_for_stability
                    )
                )
                consecutive_stable_trials = (
                    consecutive_stable_trials + 1 if stable else 0
                )
                maximum_consecutive_stable_trials = max(
                    maximum_consecutive_stable_trials,
                    consecutive_stable_trials,
                )
                if (
                    first_stability_trial is None
                    and consecutive_stable_trials
                    >= self.config.stable_trials_required
                ):
                    first_stability_trial = trial_number
                    first_stability_elapsed_seconds = (
                        time.perf_counter() - started_at
                    )
                if self.config.stop_when_solved:
                    try:
                        self._label_trial_path(
                            trial_result.visited_od_states,
                            deadline=deadline,
                        )
                    except _DeadlineReached:
                        stop_reason = "time_limit"
                        break
                    except _MemoryLimitReached:
                        stop_reason = "memory_limit"
                        break

                    initial_od_state: ODState = (
                        self.mdp.initial_state(),
                        (),
                    )
                    if (
                        self.mdp.is_terminal(initial_od_state[0])
                        or initial_od_state in self._solved_od_states
                    ):
                        if first_solved_trial is None:
                            first_solved_trial = trial_number
                            first_solved_elapsed_seconds = (
                                time.perf_counter() - started_at
                            )
                        stop_reason = "initial_state_solved"
                        break

                if (
                    self.config.stop_when_stable
                    and consecutive_stable_trials
                    >= self.config.stable_trials_required
                ):
                    stop_reason = "stable_trials"
                    break
                self._check_deadline(deadline)
        finally:
            snapshot = monitor.stop()
            self._resource_monitor = None

        visited_real_states = len({state for state, _ in self.V})
        return ODRTDPPlanningResult(
            stop_reason=stop_reason,
            trials_completed=trials_completed,
            goal_reaching_trials=goal_reaching_trials,
            step_limited_trials=step_limited_trials,
            total_real_steps=total_real_steps,
            elapsed_seconds=time.perf_counter() - started_at,
            bellman_backups=self.bellman_backups,
            planning_operator_evaluations=self.planning_operator_evaluations,
            complete_joint_actions_evaluated=(
                self.complete_joint_actions_evaluated
            ),
            transition_outcomes_evaluated=self.transition_outcomes_evaluated,
            visited_od_states=len(self.V),
            visited_real_states=visited_real_states,
            final_trial_residual=final_trial_residual,
            final_trial_scaled_residual=final_trial_scaled_residual,
            consecutive_stable_trials=consecutive_stable_trials,
            maximum_consecutive_stable_trials=(
                maximum_consecutive_stable_trials
            ),
            stability_criterion_reached=first_stability_trial is not None,
            first_stability_trial=first_stability_trial,
            first_stability_elapsed_seconds=first_stability_elapsed_seconds,
            initial_state_solved=(
                self.mdp.is_terminal(self.mdp.initial_state())
                or (self.mdp.initial_state(), ()) in self._solved_od_states
            ),
            solved_od_states=len(self._solved_od_states),
            solved_real_states=len({
                state
                for state, prefix in self._solved_od_states
                if not prefix
            }),
            solved_checks=self.solved_checks,
            first_solved_trial=first_solved_trial,
            first_solved_elapsed_seconds=first_solved_elapsed_seconds,
            resolved_max_steps_per_trial=self.resolved_max_steps_per_trial,
            memory_limit_mb=self.config.memory_limit_mb,
            memory_limit_reached=snapshot.memory_limit_reached,
            baseline_rss_mb=snapshot.baseline_rss_mb,
            peak_rss_mb=snapshot.peak_rss_mb,
            peak_rss_delta_mb=snapshot.peak_rss_delta_mb,
        )

    def reset_policy_cache_stats(self) -> None:
        """Reset OD policy-candidate cache hit/miss counters."""
        self._policy_cache_hits = 0
        self._policy_cache_misses = 0

    def policy_cache_stats(self) -> dict[str, int | float]:
        """Return complete-policy candidate-cache statistics."""
        total = self._policy_cache_hits + self._policy_cache_misses
        return {
            "hits": self._policy_cache_hits,
            "misses": self._policy_cache_misses,
            "entries": len(self._joint_policy_candidate_cache),
            "hit_rate": (self._policy_cache_hits / total if total else 0.0),
        }

    def _bellman_operator_candidates(
        self,
        od_state: ODState,
    ) -> tuple[Action, ...]:
        """Return every locally Bellman-tied operator before guidance.

        These candidates preserve the primary OD value ordering.  Secondary
        guidance is deliberately postponed when a complete evaluation action
        is built, allowing the final joint action to be compared as a whole.
        """
        self.validate_od_state(od_state)

        cached = self._raw_operator_candidate_cache.get(od_state)
        if cached is not None:
            return cached

        state, prefix = od_state
        if self.mdp.is_terminal(state):
            terminal_candidates = ("stay",)
            self._raw_operator_candidate_cache[od_state] = (
                terminal_candidates
            )
            return terminal_candidates

        current_agent = len(prefix)
        if (
            self.mdp.config.freeze_agents_at_goal
            and state[current_agent] == self.mdp.goals[current_agent]
        ):
            # Every label has exactly the same physical effect for a frozen
            # agent. Canonicalizing it to stay removes 5-way artificial ties.
            frozen_candidates = ("stay",)
            self._raw_operator_candidate_cache[od_state] = frozen_candidates
            return frozen_candidates

        best_value = math.inf
        best_actions: list[Action] = []

        for action in ACTIONS:
            candidate_value = self.operator_value(
                od_state,
                action,
                count_metrics=False,
                deadline=None,
            )

            if candidate_value < best_value and not self._values_tied(candidate_value, best_value):
                best_value = candidate_value
                best_actions = [action]
            elif self._values_tied(candidate_value, best_value):
                best_actions.append(action)

        if not best_actions:
            raise RuntimeError("No OD operator was generated")

        candidates = tuple(best_actions)
        self._raw_operator_candidate_cache[od_state] = candidates
        return candidates

    def greedy_operator_candidates(
        self,
        od_state: ODState,
    ) -> tuple[Action, ...]:
        """
        Return all locally greedy operators for one incomplete OD state.

        Candidate sets are memoized because V is fixed during evaluation. The
        selected operator itself is not cached, allowing stochastic tie
        breaking whenever the same real state and prefix are revisited.
        """
        self.validate_od_state(od_state)

        cached = self._guided_operator_candidate_cache.get(od_state)
        if cached is not None:
            return cached

        best_actions = self._guided_operators(
            od_state,
            list(self._bellman_operator_candidates(od_state)),
        )

        candidates = tuple(best_actions)
        self._guided_operator_candidate_cache[od_state] = candidates
        return candidates

    def _guided_complete_joint_actions(
        self,
        state: State,
        candidates: list[JointAction],
    ) -> list[JointAction]:
        """Refine complete OD actions using one global secondary key."""
        if len(candidates) <= 1:
            return candidates

        guidance_method = getattr(
            self.heuristic,
            "joint_action_guidance_key",
            None,
        )

        if not callable(guidance_method):
            return candidates

        keyed_candidates = [
            (guidance_method(state, action), action)
            for action in candidates
        ]
        best_key = min(key for key, _ in keyed_candidates)
        return [
            action
            for key, action in keyed_candidates
            if key == best_key
        ]

    def greedy_joint_action_candidates(
        self,
        state: State,
    ) -> tuple[JointAction, ...]:
        """Return globally guided complete actions induced by OD ties.

        The search expands every Bellman-tied local branch and compares the
        resulting complete joint actions with one global secondary key.
        """
        self.mdp.validate_state(state)

        cached = self._joint_policy_candidate_cache.get(state)
        if cached is not None:
            self._policy_cache_hits += 1
            return cached

        self._policy_cache_misses += 1

        if self.mdp.is_terminal(state):
            terminal_candidates = (
                tuple("stay" for _ in range(self.mdp.n_agents)),
            )
            self._joint_policy_candidate_cache[state] = terminal_candidates
            return terminal_candidates

        def expand(*, locally_guided: bool) -> list[JointAction]:
            partial_prefixes: list[JointAction] = [()]

            while (
                partial_prefixes
                and len(partial_prefixes[0]) < self.mdp.n_agents
            ):
                expanded: list[JointAction] = []
                for prefix in partial_prefixes:
                    od_state = (state, prefix)
                    local_candidates = (
                        self.greedy_operator_candidates(od_state)
                        if locally_guided
                        else self._bellman_operator_candidates(od_state)
                    )
                    expanded.extend(
                        prefix + (action,)
                        for action in local_candidates
                    )

                partial_prefixes = expanded

            return partial_prefixes

        partial_prefixes = expand(locally_guided=False)

        if not partial_prefixes:
            raise RuntimeError("No complete OD joint action was generated")

        guided = self._guided_complete_joint_actions(
            state,
            partial_prefixes,
        )
        candidates = tuple(guided)
        self._joint_policy_candidate_cache[state] = candidates
        return candidates

    def _deterministic_joint_tie_choice(
        self,
        candidates: list[JointAction],
        state: State,
    ) -> JointAction:
        """Choose reproducibly among complete globally guided actions."""
        if not candidates:
            raise ValueError("candidates cannot be empty")

        payload = repr(
            (
                self.config.seed,
                state,
                tuple(candidates),
                "complete-od-policy",
            )
        ).encode("utf-8")
        digest = hashlib.sha256(payload).digest()
        index = int.from_bytes(digest[:8], byteorder="big") % len(candidates)
        return candidates[index]


    def global_real_greedy_action_candidates(
        self,
        state: State,
    ) -> tuple[JointAction, ...]:
        """Diagnostic policy extracted directly from real-state OD values.

        This enumerates all complete joint actions and evaluates them with
        ``V_OD(next_state, empty_prefix)``.  Comparing it with the ordinary
        prefix-induced policy separates value-learning errors from policy-
        extraction errors.
        """
        self.mdp.validate_state(state)
        cached = self._global_real_policy_candidate_cache.get(state)
        if cached is not None:
            return cached
        if self.mdp.is_terminal(state):
            result = (tuple("stay" for _ in range(self.mdp.n_agents)),)
            self._global_real_policy_candidate_cache[state] = result
            return result

        best_value = math.inf
        best_actions: list[JointAction] = []
        for joint_action in self.mdp.all_joint_actions():
            value = self.complete_joint_action_value(
                state, joint_action, count_metrics=False, deadline=None
            )
            if value < best_value and not self._values_tied(value, best_value):
                best_value = value
                best_actions = [joint_action]
            elif self._values_tied(value, best_value):
                best_actions.append(joint_action)
        guided = self._guided_complete_joint_actions(state, best_actions)
        result = tuple(guided)
        self._global_real_policy_candidate_cache[state] = result
        return result

    def global_policy_action_with_info(
        self,
        state: State,
        *,
        tie_rng: random.Random | None = None,
    ) -> tuple[JointAction, int]:
        candidates = self.global_real_greedy_action_candidates(state)
        if len(candidates) == 1:
            return candidates[0], 0
        if tie_rng is not None:
            return tie_rng.choice(candidates), 1
        return self._deterministic_joint_tie_choice(
            list(candidates), state
        ), 1

    def global_policy_action(
        self,
        state: State,
        *,
        tie_rng: random.Random | None = None,
    ) -> JointAction:
        return self.global_policy_action_with_info(
            state, tie_rng=tie_rng
        )[0]

    def policy_action_with_info(
        self,
        state: State,
        *,
        tie_rng: random.Random | None = None,
    ) -> tuple[JointAction, int]:
        """
        Select one globally guided complete OD action.

        The returned tie count retains the old interpretation: it counts the
        number of prefixes on the selected path that had multiple Bellman-tied
        local operators before secondary guidance.
        """
        self.mdp.validate_state(state)

        if self.mdp.is_terminal(state):
            return (
                tuple("stay" for _ in range(self.mdp.n_agents)),
                0,
            )

        candidates = self.greedy_joint_action_candidates(state)

        if len(candidates) == 1:
            selected = candidates[0]
        elif tie_rng is not None:
            selected = tie_rng.choice(candidates)
        else:
            selected = self._deterministic_joint_tie_choice(
                list(candidates),
                state,
            )

        tie_decisions = 0
        for prefix_length in range(self.mdp.n_agents):
            prefix = selected[:prefix_length]
            if len(
                self._bellman_operator_candidates((state, prefix))
            ) > 1:
                tie_decisions += 1

        return selected, tie_decisions

    def policy_action(
        self,
        state: State,
        *,
        tie_rng: random.Random | None = None,
    ) -> JointAction:
        """
        Construct a globally guided current greedy joint action without backups.

        Supplying ``tie_rng`` samples among complete actions that remain tied
        after global guidance. Candidate sets are cached per real state.
        """
        action, _ = self.policy_action_with_info(
            state,
            tie_rng=tie_rng,
        )
        return action



"""Fixed-policy evaluation for Baseline RTDP and OD-RTDP.

Evaluation never changes the planners' value tables. The optimized path keeps
transition-cache reads enabled while suppressing writes during the expensive
search over candidate actions. After one action is selected, only that action's
transition distribution is memoized and sampled.
"""

from collections import Counter
from dataclasses import asdict, dataclass
import math
import random
import statistics
import time
from typing import Any, Protocol



class PolicyPlanner(Protocol):
    def policy_action(
        self,
        state: State,
        *,
        tie_rng: random.Random | None = None,
    ) -> JointAction:
        ...

    def policy_action_with_info(
        self,
        state: State,
        *,
        tie_rng: random.Random | None = None,
    ) -> tuple[JointAction, int]:
        ...


class MethodPolicyAdapter:
    """Expose an alternate planner policy method through the standard protocol."""

    def __init__(
        self,
        planner: Any,
        *,
        action_method: str,
        info_method: str,
        policy_name: str,
    ) -> None:
        self._planner = planner
        self.mdp = planner.mdp
        self.heuristic = getattr(planner, "heuristic", None)
        self.resolved_max_steps_per_trial = getattr(
            planner, "resolved_max_steps_per_trial", None
        )
        self._action_method = action_method
        self._info_method = info_method
        self.policy_name = policy_name

    def policy_action(self, state: State, *, tie_rng=None) -> JointAction:
        return getattr(self._planner, self._action_method)(
            state, tie_rng=tie_rng
        )

    def policy_action_with_info(self, state: State, *, tie_rng=None):
        return getattr(self._planner, self._info_method)(
            state, tie_rng=tie_rng
        )

    def reset_policy_cache_stats(self) -> None:
        method = getattr(self._planner, "reset_policy_cache_stats", None)
        if callable(method):
            method()

    def policy_cache_stats(self):
        method = getattr(self._planner, "policy_cache_stats", None)
        return method() if callable(method) else {
            "hits": 0, "misses": 0, "entries": 0, "hit_rate": 0.0
        }


@dataclass(frozen=True)
class EvaluationConfig:
    """Configuration for evaluating one fixed value function."""

    episodes: int = 100
    seed: int = 0
    max_steps_per_episode: int | None = None
    measure_conflict_risk: bool = True
    randomize_greedy_ties: bool = False

    # Main optimization: policy extraction may read transitions produced in
    # planning, but it does not cache every rejected candidate action. The
    # selected action is cached immediately afterwards when it is executed.
    cache_only_executed_actions: bool = True

    # Cycle/tie/action-quality diagnostics are useful for pilot runs but can be
    # disabled for the fastest large experiment.
    collect_diagnostics: bool = True

    def __post_init__(self) -> None:
        if self.episodes <= 0:
            raise ValueError("episodes must be positive")
        if (
            self.max_steps_per_episode is not None
            and self.max_steps_per_episode <= 0
        ):
            raise ValueError(
                "max_steps_per_episode must be positive or None"
            )


@dataclass(frozen=True)
class EpisodeResult:
    episode_index: int
    episode_seed: int
    success: bool
    hit_step_limit: bool
    steps: int
    accumulated_cost: float
    makespan: int | None
    arrival_times: tuple[int | None, ...]
    arrived_agents: int
    cumulative_conflict_risk: float
    mean_conflict_risk_per_step: float
    policy_decision_seconds: float
    episode_elapsed_seconds: float

    unique_states_visited: int
    repeated_state_visits: int
    maximum_state_visit_count: int
    self_transitions: int
    maximum_consecutive_self_transitions: int
    tie_decisions: int
    unique_real_states_with_policy_ties: int

    cumulative_expected_self_loop_probability: float
    mean_expected_self_loop_probability_per_step: float
    cumulative_expected_shortest_path_progress: float
    mean_expected_shortest_path_progress_per_step: float

    cumulative_expected_vertex_conflict_probability: float
    cumulative_expected_edge_swap_probability: float
    cumulative_expected_noncollision_no_motion_probability: float
    mean_expected_vertex_conflict_probability_per_step: float
    mean_expected_edge_swap_probability_per_step: float
    mean_expected_noncollision_no_motion_probability_per_step: float
    selected_unfinished_stay_actions: int
    selected_unfinished_blocked_actions: int
    deterministic_self_loop_actions: int
    failure_reason: str | None
    repeated_state_action_pairs: tuple[dict[str, Any], ...]

    final_state: State

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


@dataclass(frozen=True)
class EvaluationSummary:
    policy_name: str
    map_name: str
    scenario_name: str
    n_agents: int
    evaluation_seed: int
    episodes: int
    max_steps_per_episode: int
    randomize_greedy_ties: bool
    cache_only_executed_actions: bool
    collect_diagnostics: bool

    successful_episodes: int
    failed_episodes: int
    success_rate: float
    total_environment_steps: int
    mean_steps_all_episodes: float
    mean_steps_successful_episodes: float | None
    mean_accumulated_cost_all_episodes: float
    mean_sum_of_costs_successful_episodes: float | None
    std_sum_of_costs_successful_episodes: float | None
    mean_makespan_successful_episodes: float | None
    std_makespan_successful_episodes: float | None
    per_agent_arrival_rates: tuple[float, ...]
    per_agent_mean_arrival_times: tuple[float | None, ...]
    mean_arrived_agents_per_episode: float

    expected_conflict_attempts_per_episode: float
    mean_conflict_risk_per_environment_step: float
    mean_expected_self_loop_probability_per_step: float
    mean_expected_shortest_path_progress_per_step: float
    mean_expected_vertex_conflict_probability_per_step: float
    mean_expected_edge_swap_probability_per_step: float
    mean_expected_noncollision_no_motion_probability_per_step: float
    mean_selected_unfinished_stay_actions_per_episode: float
    mean_selected_unfinished_blocked_actions_per_episode: float
    deterministic_self_loop_failures: int
    step_limit_failures: int

    evaluation_elapsed_seconds: float
    evaluation_baseline_rss_mb: float
    evaluation_peak_rss_mb: float
    evaluation_peak_rss_delta_mb: float
    total_policy_decision_seconds: float
    mean_policy_decision_milliseconds: float

    policy_cache_hits: int
    policy_cache_misses: int
    policy_cache_entries: int
    policy_cache_hit_rate: float

    transition_raw_cache_entries_before: int
    transition_raw_cache_entries_after: int
    transition_resolved_cache_entries_before: int
    transition_resolved_cache_entries_after: int
    transition_raw_cache_hits: int
    transition_raw_cache_misses: int
    transition_raw_cache_writes: int
    transition_raw_cache_evictions: int
    transition_resolved_cache_hits: int
    transition_resolved_cache_misses: int
    transition_resolved_cache_writes: int
    transition_resolved_cache_evictions: int

    mean_unique_states_visited: float
    mean_repeated_state_visits: float
    mean_maximum_state_visit_count: float
    mean_self_transitions: float
    mean_maximum_consecutive_self_transitions: float
    mean_tie_decisions: float
    mean_unique_real_states_with_policy_ties: float
    mean_tie_decisions_per_environment_step: float

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


@dataclass(frozen=True)
class EvaluationResult:
    config: EvaluationConfig
    summary: EvaluationSummary
    episode_results: tuple[EpisodeResult, ...]

    def summary_dict(self) -> dict[str, Any]:
        return self.summary.to_dict()


def _mean_or_none(values: list[float] | list[int]) -> float | None:
    return float(statistics.fmean(values)) if values else None


def _sample_std_or_none(values: list[float] | list[int]) -> float | None:
    if not values:
        return None
    if len(values) == 1:
        return 0.0
    return float(statistics.stdev(values))


def _resolve_episode_step_limit(
    planner: PolicyPlanner,
    config: EvaluationConfig,
) -> int:
    if config.max_steps_per_episode is not None:
        return config.max_steps_per_episode

    planner_limit = getattr(planner, "resolved_max_steps_per_trial", None)
    if planner_limit is None:
        raise ValueError(
            "Evaluation requires a finite step limit. Supply "
            "EvaluationConfig(max_steps_per_episode=...)."
        )
    if not isinstance(planner_limit, int) or planner_limit <= 0:
        raise ValueError(
            "planner.resolved_max_steps_per_trial must be a positive integer"
        )
    return planner_limit


def _validate_planner_environment(
    mdp: GridMMDP,
    planner: PolicyPlanner,
) -> None:
    planner_mdp = getattr(planner, "mdp", None)
    if planner_mdp is not None and planner_mdp is not mdp:
        raise ValueError(
            "The planner belongs to a different GridMMDP object."
        )


def _policy_action_with_info(
    planner: PolicyPlanner,
    state: State,
    tie_rng: random.Random | None,
    collect_diagnostics: bool,
) -> tuple[JointAction, int]:
    if collect_diagnostics:
        method = getattr(planner, "policy_action_with_info", None)
        if callable(method):
            return method(state, tie_rng=tie_rng)

    policy_method = getattr(planner, "policy_action")
    try:
        return policy_method(state, tie_rng=tie_rng), 0
    except TypeError:
        return policy_method(state), 0


def _distance_sum(planner: PolicyPlanner, state: State) -> float | None:
    heuristic = getattr(planner, "heuristic", None)
    method = getattr(heuristic, "distance_summary", None)
    if not callable(method):
        return None
    distances = method(state)
    if any(math.isinf(distance) for distance in distances):
        return None
    return float(sum(distances))


def _selected_action_diagnostics(
    planner: PolicyPlanner,
    state: State,
    transitions: tuple[Transition, ...],
) -> tuple[float, float]:
    """Return expected self-loop probability and shortest-path progress."""
    self_loop_probability = sum(
        probability
        for next_state, probability in transitions
        if next_state == state
    )

    current_distance = _distance_sum(planner, state)
    if current_distance is None:
        return self_loop_probability, 0.0

    expected_next_distance = 0.0
    for next_state, probability in transitions:
        next_distance = _distance_sum(planner, next_state)
        if next_distance is None:
            return self_loop_probability, 0.0
        expected_next_distance += probability * next_distance

    return self_loop_probability, current_distance - expected_next_distance


def evaluate_episode(
    mdp: GridMMDP,
    planner: PolicyPlanner,
    *,
    episode_index: int,
    episode_seed: int,
    max_steps: int,
    measure_conflict_risk: bool = True,
    randomize_greedy_ties: bool = False,
    cache_only_executed_actions: bool = True,
    collect_diagnostics: bool = True,
) -> EpisodeResult:
    if episode_index < 0:
        raise ValueError("episode_index cannot be negative")
    if max_steps <= 0:
        raise ValueError("max_steps must be positive")

    _validate_planner_environment(mdp, planner)
    episode_started_at = time.perf_counter()

    transition_rng = random.Random(episode_seed)
    tie_rng = (
        random.Random(episode_seed ^ 0xD1B54A32D192ED03)
        if randomize_greedy_ties
        else None
    )

    state = mdp.initial_state()
    steps = 0
    accumulated_cost = 0.0
    arrival_times: list[int | None] = [
        0 if position == goal else None
        for position, goal in zip(state, mdp.goals)
    ]

    cumulative_conflict_risk = 0.0
    policy_decision_seconds = 0.0
    cumulative_expected_self_loop_probability = 0.0
    cumulative_expected_shortest_path_progress = 0.0
    cumulative_expected_vertex_conflict_probability = 0.0
    cumulative_expected_edge_swap_probability = 0.0
    cumulative_expected_noncollision_no_motion_probability = 0.0
    selected_unfinished_stay_actions = 0
    selected_unfinished_blocked_actions = 0
    deterministic_self_loop_actions = 0
    failure_reason: str | None = None
    state_action_counts: Counter[tuple[State, JointAction]] | None = (
        Counter() if collect_diagnostics else None
    )

    visit_counts: Counter[State] | None = (
        Counter({state: 1}) if collect_diagnostics else None
    )
    repeated_state_visits = 0
    self_transitions = 0
    current_self_transition_streak = 0
    maximum_consecutive_self_transitions = 0
    tie_decisions = 0
    real_states_with_ties: set[State] | None = (
        set() if collect_diagnostics else None
    )

    while not mdp.is_terminal(state) and steps < max_steps:
        decision_started_at = time.perf_counter()
        with mdp.transition_cache_writes(
            not cache_only_executed_actions
        ):
            joint_action, step_tie_decisions = _policy_action_with_info(
                planner,
                state,
                tie_rng,
                collect_diagnostics,
            )
        policy_decision_seconds += time.perf_counter() - decision_started_at

        if collect_diagnostics:
            tie_decisions += step_tie_decisions
            if step_tie_decisions > 0 and real_states_with_ties is not None:
                real_states_with_ties.add(state)

        mdp.validate_joint_action(joint_action)

        # Writes are enabled again here. Thus only the action actually used by
        # the environment is added to the transition cache.
        transitions = mdp.joint_transitions(state, joint_action)

        if measure_conflict_risk:
            cumulative_conflict_risk += mdp.conflict_probability(
                state,
                joint_action,
            )

        if collect_diagnostics:
            self_loop_probability, expected_progress = (
                _selected_action_diagnostics(
                    planner,
                    state,
                    transitions,
                )
            )
            cumulative_expected_self_loop_probability += self_loop_probability
            cumulative_expected_shortest_path_progress += expected_progress
            breakdown = mdp.action_risk_breakdown(state, joint_action)
            cumulative_expected_vertex_conflict_probability += float(
                breakdown["vertex_conflict_probability"]
            )
            cumulative_expected_edge_swap_probability += float(
                breakdown["edge_swap_probability"]
            )
            cumulative_expected_noncollision_no_motion_probability += float(
                breakdown["noncollision_no_motion_probability"]
            )
            selected_unfinished_stay_actions += int(
                breakdown["unfinished_stay_actions"]
            )
            selected_unfinished_blocked_actions += int(
                breakdown["unfinished_blocked_actions"]
            )
            if state_action_counts is not None:
                state_action_counts[(state, joint_action)] += 1
            if (
                tie_rng is None
                and self_loop_probability >= 1.0 - 1e-15
                and not mdp.is_terminal(state)
            ):
                deterministic_self_loop_actions += 1
                failure_reason = "deterministic_self_loop_policy"
                break

        next_state = mdp.sample_from_transitions(
            transitions,
            transition_rng,
        )

        accumulated_cost += mdp.transition_cost(
            state,
            joint_action,
            next_state,
        )
        steps += 1

        for agent_index, (next_position, goal) in enumerate(
            zip(next_state, mdp.goals)
        ):
            if (
                arrival_times[agent_index] is None
                and next_position == goal
            ):
                arrival_times[agent_index] = steps

        if collect_diagnostics and visit_counts is not None:
            if next_state in visit_counts:
                repeated_state_visits += 1
            visit_counts[next_state] += 1

            if next_state == state:
                self_transitions += 1
                current_self_transition_streak += 1
                maximum_consecutive_self_transitions = max(
                    maximum_consecutive_self_transitions,
                    current_self_transition_streak,
                )
            else:
                current_self_transition_streak = 0

        state = next_state

    success = mdp.is_terminal(state)
    hit_step_limit = not success and steps >= max_steps
    if not success and failure_reason is None:
        failure_reason = "step_limit" if hit_step_limit else "terminated"
    arrived_agents = sum(value is not None for value in arrival_times)
    makespan = (
        max(value for value in arrival_times if value is not None)
        if success
        else None
    )

    if success and mdp.config.freeze_agents_at_goal:
        arrival_time_sum = float(
            sum(value for value in arrival_times if value is not None)
        )
        if not math.isclose(
            accumulated_cost,
            arrival_time_sum,
            rel_tol=0.0,
            abs_tol=1e-9,
        ):
            raise RuntimeError(
                "Accumulated cost does not equal the sum of arrival times."
            )

    unique_states = len(visit_counts) if visit_counts is not None else 0
    maximum_state_visit_count = (
        max(visit_counts.values()) if visit_counts else 0
    )

    repeated_pairs: tuple[dict[str, Any], ...] = ()
    if state_action_counts is not None:
        repeated_pairs = tuple(
            {
                "state": state_key,
                "action": action_key,
                "count": count,
                "risk": mdp.action_risk_breakdown(state_key, action_key),
            }
            for (state_key, action_key), count in sorted(
                state_action_counts.items(),
                key=lambda item: (-item[1], repr(item[0])),
            )
            if count > 1
        )

    return EpisodeResult(
        episode_index=episode_index,
        episode_seed=episode_seed,
        success=success,
        hit_step_limit=hit_step_limit,
        steps=steps,
        accumulated_cost=accumulated_cost,
        makespan=makespan,
        arrival_times=tuple(arrival_times),
        arrived_agents=arrived_agents,
        cumulative_conflict_risk=cumulative_conflict_risk,
        mean_conflict_risk_per_step=(
            cumulative_conflict_risk / steps if steps else 0.0
        ),
        policy_decision_seconds=policy_decision_seconds,
        episode_elapsed_seconds=time.perf_counter() - episode_started_at,
        unique_states_visited=unique_states,
        repeated_state_visits=repeated_state_visits,
        maximum_state_visit_count=maximum_state_visit_count,
        self_transitions=self_transitions,
        maximum_consecutive_self_transitions=(
            maximum_consecutive_self_transitions
        ),
        tie_decisions=tie_decisions,
        unique_real_states_with_policy_ties=(
            len(real_states_with_ties) if real_states_with_ties is not None else 0
        ),
        cumulative_expected_self_loop_probability=(
            cumulative_expected_self_loop_probability
        ),
        mean_expected_self_loop_probability_per_step=(
            cumulative_expected_self_loop_probability / steps if steps else 0.0
        ),
        cumulative_expected_shortest_path_progress=(
            cumulative_expected_shortest_path_progress
        ),
        mean_expected_shortest_path_progress_per_step=(
            cumulative_expected_shortest_path_progress / steps if steps else 0.0
        ),
        cumulative_expected_vertex_conflict_probability=(
            cumulative_expected_vertex_conflict_probability
        ),
        cumulative_expected_edge_swap_probability=(
            cumulative_expected_edge_swap_probability
        ),
        cumulative_expected_noncollision_no_motion_probability=(
            cumulative_expected_noncollision_no_motion_probability
        ),
        mean_expected_vertex_conflict_probability_per_step=(
            cumulative_expected_vertex_conflict_probability / steps if steps else 0.0
        ),
        mean_expected_edge_swap_probability_per_step=(
            cumulative_expected_edge_swap_probability / steps if steps else 0.0
        ),
        mean_expected_noncollision_no_motion_probability_per_step=(
            cumulative_expected_noncollision_no_motion_probability / steps if steps else 0.0
        ),
        selected_unfinished_stay_actions=selected_unfinished_stay_actions,
        selected_unfinished_blocked_actions=selected_unfinished_blocked_actions,
        deterministic_self_loop_actions=deterministic_self_loop_actions,
        failure_reason=failure_reason,
        repeated_state_action_pairs=repeated_pairs,
        final_state=state,
    )


def _planner_cache_stats(planner: PolicyPlanner) -> dict[str, int | float]:
    method = getattr(planner, "policy_cache_stats", None)
    if not callable(method):
        return {"hits": 0, "misses": 0, "entries": 0, "hit_rate": 0.0}
    return method()


def _build_summary(
    mdp: GridMMDP,
    planner: PolicyPlanner,
    config: EvaluationConfig,
    max_steps_per_episode: int,
    episode_results: tuple[EpisodeResult, ...],
    evaluation_elapsed_seconds: float,
    transition_stats_before: dict[str, int | bool | None],
    transition_stats_after: dict[str, int | bool | None],
    resource_snapshot: Any,
) -> EvaluationSummary:
    successful = [result for result in episode_results if result.success]
    successful_costs = [result.accumulated_cost for result in successful]
    successful_steps = [result.steps for result in successful]
    successful_makespans = [
        result.makespan
        for result in successful
        if result.makespan is not None
    ]

    total_environment_steps = sum(result.steps for result in episode_results)
    total_policy_decision_seconds = sum(
        result.policy_decision_seconds for result in episode_results
    )
    total_conflict_risk = sum(
        result.cumulative_conflict_risk for result in episode_results
    )
    total_tie_decisions = sum(
        result.tie_decisions for result in episode_results
    )
    total_expected_self_loop = sum(
        result.cumulative_expected_self_loop_probability
        for result in episode_results
    )
    total_expected_progress = sum(
        result.cumulative_expected_shortest_path_progress
        for result in episode_results
    )
    total_vertex = sum(
        result.cumulative_expected_vertex_conflict_probability
        for result in episode_results
    )
    total_edge = sum(
        result.cumulative_expected_edge_swap_probability
        for result in episode_results
    )
    total_no_motion = sum(
        result.cumulative_expected_noncollision_no_motion_probability
        for result in episode_results
    )

    per_agent_arrival_rates: list[float] = []
    per_agent_mean_arrival_times: list[float | None] = []
    for agent_index in range(mdp.n_agents):
        times = [
            result.arrival_times[agent_index]
            for result in episode_results
            if result.arrival_times[agent_index] is not None
        ]
        per_agent_arrival_rates.append(len(times) / config.episodes)
        per_agent_mean_arrival_times.append(_mean_or_none(times))

    policy_stats = _planner_cache_stats(planner)
    successful_episodes = len(successful)

    return EvaluationSummary(
        policy_name=getattr(planner, "policy_name", type(planner).__name__),
        map_name=mdp.map_name,
        scenario_name=mdp.instance.scenario_file.name,
        n_agents=mdp.n_agents,
        evaluation_seed=config.seed,
        episodes=config.episodes,
        max_steps_per_episode=max_steps_per_episode,
        randomize_greedy_ties=config.randomize_greedy_ties,
        cache_only_executed_actions=config.cache_only_executed_actions,
        collect_diagnostics=config.collect_diagnostics,
        successful_episodes=successful_episodes,
        failed_episodes=config.episodes - successful_episodes,
        success_rate=successful_episodes / config.episodes,
        total_environment_steps=total_environment_steps,
        mean_steps_all_episodes=float(
            statistics.fmean(result.steps for result in episode_results)
        ),
        mean_steps_successful_episodes=_mean_or_none(successful_steps),
        mean_accumulated_cost_all_episodes=float(
            statistics.fmean(
                result.accumulated_cost for result in episode_results
            )
        ),
        mean_sum_of_costs_successful_episodes=(
            _mean_or_none(successful_costs)
        ),
        std_sum_of_costs_successful_episodes=(
            _sample_std_or_none(successful_costs)
        ),
        mean_makespan_successful_episodes=(
            _mean_or_none(successful_makespans)
        ),
        std_makespan_successful_episodes=(
            _sample_std_or_none(successful_makespans)
        ),
        per_agent_arrival_rates=tuple(per_agent_arrival_rates),
        per_agent_mean_arrival_times=tuple(per_agent_mean_arrival_times),
        mean_arrived_agents_per_episode=float(
            statistics.fmean(
                result.arrived_agents for result in episode_results
            )
        ),
        expected_conflict_attempts_per_episode=float(
            statistics.fmean(
                result.cumulative_conflict_risk for result in episode_results
            )
        ),
        mean_conflict_risk_per_environment_step=(
            total_conflict_risk / total_environment_steps
            if total_environment_steps
            else 0.0
        ),
        mean_expected_self_loop_probability_per_step=(
            total_expected_self_loop / total_environment_steps
            if total_environment_steps
            else 0.0
        ),
        mean_expected_shortest_path_progress_per_step=(
            total_expected_progress / total_environment_steps
            if total_environment_steps
            else 0.0
        ),
        mean_expected_vertex_conflict_probability_per_step=(
            total_vertex / total_environment_steps if total_environment_steps else 0.0
        ),
        mean_expected_edge_swap_probability_per_step=(
            total_edge / total_environment_steps if total_environment_steps else 0.0
        ),
        mean_expected_noncollision_no_motion_probability_per_step=(
            total_no_motion / total_environment_steps if total_environment_steps else 0.0
        ),
        mean_selected_unfinished_stay_actions_per_episode=float(
            statistics.fmean(r.selected_unfinished_stay_actions for r in episode_results)
        ),
        mean_selected_unfinished_blocked_actions_per_episode=float(
            statistics.fmean(r.selected_unfinished_blocked_actions for r in episode_results)
        ),
        deterministic_self_loop_failures=sum(
            r.failure_reason == "deterministic_self_loop_policy" for r in episode_results
        ),
        step_limit_failures=sum(
            r.failure_reason == "step_limit" for r in episode_results
        ),
        evaluation_elapsed_seconds=evaluation_elapsed_seconds,
        evaluation_baseline_rss_mb=resource_snapshot.baseline_rss_mb,
        evaluation_peak_rss_mb=resource_snapshot.peak_rss_mb,
        evaluation_peak_rss_delta_mb=resource_snapshot.peak_rss_delta_mb,
        total_policy_decision_seconds=total_policy_decision_seconds,
        mean_policy_decision_milliseconds=(
            1_000.0 * total_policy_decision_seconds / total_environment_steps
            if total_environment_steps
            else 0.0
        ),
        policy_cache_hits=int(policy_stats["hits"]),
        policy_cache_misses=int(policy_stats["misses"]),
        policy_cache_entries=int(policy_stats["entries"]),
        policy_cache_hit_rate=float(policy_stats["hit_rate"]),
        transition_raw_cache_entries_before=int(
            transition_stats_before["raw_entries"]
        ),
        transition_raw_cache_entries_after=int(
            transition_stats_after["raw_entries"]
        ),
        transition_resolved_cache_entries_before=int(
            transition_stats_before["resolved_entries"]
        ),
        transition_resolved_cache_entries_after=int(
            transition_stats_after["resolved_entries"]
        ),
        transition_raw_cache_hits=int(transition_stats_after["raw_hits"]),
        transition_raw_cache_misses=int(transition_stats_after["raw_misses"]),
        transition_raw_cache_writes=int(transition_stats_after["raw_writes"]),
        transition_raw_cache_evictions=int(
            transition_stats_after.get("raw_evictions", 0)
        ),
        transition_resolved_cache_hits=int(
            transition_stats_after["resolved_hits"]
        ),
        transition_resolved_cache_misses=int(
            transition_stats_after["resolved_misses"]
        ),
        transition_resolved_cache_writes=int(
            transition_stats_after["resolved_writes"]
        ),
        transition_resolved_cache_evictions=int(
            transition_stats_after.get("resolved_evictions", 0)
        ),
        mean_unique_states_visited=float(
            statistics.fmean(
                result.unique_states_visited for result in episode_results
            )
        ),
        mean_repeated_state_visits=float(
            statistics.fmean(
                result.repeated_state_visits for result in episode_results
            )
        ),
        mean_maximum_state_visit_count=float(
            statistics.fmean(
                result.maximum_state_visit_count for result in episode_results
            )
        ),
        mean_self_transitions=float(
            statistics.fmean(
                result.self_transitions for result in episode_results
            )
        ),
        mean_maximum_consecutive_self_transitions=float(
            statistics.fmean(
                result.maximum_consecutive_self_transitions
                for result in episode_results
            )
        ),
        mean_tie_decisions=float(
            statistics.fmean(
                result.tie_decisions for result in episode_results
            )
        ),
        mean_unique_real_states_with_policy_ties=float(
            statistics.fmean(
                result.unique_real_states_with_policy_ties
                for result in episode_results
            )
        ),
        mean_tie_decisions_per_environment_step=(
            total_tie_decisions / total_environment_steps
            if total_environment_steps
            else 0.0
        ),
    )


def evaluate_policy(
    mdp: GridMMDP,
    planner: PolicyPlanner,
    config: EvaluationConfig | None = None,
) -> EvaluationResult:
    """Evaluate a fixed planner value function over stochastic episodes."""
    evaluation_config = config if config is not None else EvaluationConfig()
    _validate_planner_environment(mdp, planner)
    max_steps_per_episode = _resolve_episode_step_limit(
        planner,
        evaluation_config,
    )

    reset_policy_stats = getattr(planner, "reset_policy_cache_stats", None)
    if callable(reset_policy_stats):
        reset_policy_stats()

    transition_stats_before = mdp.transition_cache_stats()
    mdp.reset_transition_cache_stats()

    master_rng = random.Random(evaluation_config.seed)
    episode_seeds = [
        master_rng.randrange(0, 2**63)
        for _ in range(evaluation_config.episodes)
    ]

    evaluation_started_at = time.perf_counter()
    monitor = ResourceMonitor().start()
    try:
        episode_results = tuple(
            evaluate_episode(
                mdp,
                planner,
                episode_index=episode_index,
                episode_seed=episode_seed,
                max_steps=max_steps_per_episode,
                measure_conflict_risk=evaluation_config.measure_conflict_risk,
                randomize_greedy_ties=evaluation_config.randomize_greedy_ties,
                cache_only_executed_actions=(
                    evaluation_config.cache_only_executed_actions
                ),
                collect_diagnostics=evaluation_config.collect_diagnostics,
            )
            for episode_index, episode_seed in enumerate(episode_seeds)
        )
    finally:
        resource_snapshot = monitor.stop()
    evaluation_elapsed_seconds = time.perf_counter() - evaluation_started_at
    transition_stats_after = mdp.transition_cache_stats()

    summary = _build_summary(
        mdp,
        planner,
        evaluation_config,
        max_steps_per_episode,
        episode_results,
        evaluation_elapsed_seconds,
        transition_stats_before,
        transition_stats_after,
        resource_snapshot,
    )

    return EvaluationResult(
        config=evaluation_config,
        summary=summary,
        episode_results=episode_results,
    )




### 1. Environment Setup
We load a map and configure the slip probability.

In [ ]:
# Load a 9x9 crossing diagnostic map with 3 agents
map_instance = create_map_instance("maps/diagnostic/crossing-9-9", scenario_number=1, task_offset=0, n_agents=3)

mdp_config = MMDPConfig(slip_to_stay_probability=0.20)
mdp = GridMMDP(map_instance, mdp_config)

heuristic = ShortestPathHeuristic(mdp)

print(f"Map loaded: {map_instance.map_name}")
print(f"Number of agents: {mdp.config_agents}")

### 2. Baseline RTDP

In [ ]:
baseline_config = RTDPConfig(time_limit_seconds=5.0, seed=42)
baseline_planner = BaselineRTDP(mdp, heuristic, baseline_config)
baseline_result = baseline_planner.solve()
print(f"Baseline Stop reason: {baseline_result.stop_reason}")
print(f"Baseline Bellman backups: {baseline_result.bellman_backups}")

### 3. OD RTDP

In [ ]:
od_config = RTDPConfig(time_limit_seconds=5.0, seed=42)
od_planner = OperatorDecompositionRTDP(mdp, heuristic, od_config)
od_result = od_planner.solve()
print(f"OD Stop reason: {od_result.stop_reason}")
print(f"OD Bellman backups: {od_result.bellman_backups}")